# Canadian Studies

---
BC 
---

* load packages

In [ ]:
import warnings
from pathlib import Path
import textwrap
import geopandas as gpd
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rioxarray as rxr
from IPython.display import display

import RES.visuals as vis
from RES import lands
from RES import utility as utils
from RES.hdf5_handler import DataHandler

# Setting plotting defaults to Elsevier style
plt.style.use('../RES/visual_styles/elsevier.mplstyle')
# Suppress specific warnings
warnings.filterwarnings("ignore", category=UserWarning)
cfg_BASELINE=utils.load_config('../config/config_CAN_baseline.yaml')
cfg_policy=utils.load_config('../config/config_CAN_policy1.yaml')


CRS_m = cfg_policy.get('region_mapping').get('BC').get('CRS_meters')  # Default metric CRS
CRS_d = cfg_policy.get('default_CRS').get('degrees')  # Default geographic CRS

sub_national_unit_tag=cfg_policy.get('GADM').get('datafield_mapping').get('NAME_2') # type: ignore

# Define Province Code

In [ ]:
# Construct region_options as a list of tuples: (name, code)
# region_options = [(cfg_policy['region_mapping'][code]['name'], code) for code in cfg_policy['region_mapping']]
region_code = 'BC' 

region_name=cfg_policy.get('region_mapping').get(region_code).get('name') 
utils.print_banner(f"Selected Region: {region_name} ({region_code})")

country_name=cfg_policy.get('country','Canada') 
country_kwd=country_name.replace(' ','_')


##  Define run/scenario

In [ ]:
# Define the directory and search pattern
data_store_dir = Path(f"../data/store/{country_kwd}")
search_keyword = f"resources_{country_kwd}_{region_code}_"

# List files containing the search pattern
matching_files = [str(f) for f in data_store_dir.glob(f"*{search_keyword}*") if f.is_file()]

# Extract run IDs from matching_files
POLICYs = [f.replace(str(data_store_dir) + '/', '').replace(search_keyword, '').replace(".h5", '') for f in matching_files]
utils.print_update(level=1,message=f"Results Available for Run IDs | {region_code}: ")
utils.print_update(level=2, message="\n    ".join(POLICYs))


BASELINE_vis_save_to=Path(f'../vis/{country_kwd}/{region_code}/BASELINE')

policy_name=POLICYs[1]# cfg_policy.get('Scenario').get('run_id')
POLICY_vis_save_to=Path(f'../vis/{country_kwd}/{region_code}/{policy_name}')

BASELINE_results_save_to=Path(f'../results/{country_kwd}/{region_code}/BASELINE')
POLICY_results_save_to=Path(f'../results/{country_kwd}/{region_code}/{policy_name}')

common_vis_save_to=utils.ensure_path(f'../vis/{country_kwd}/{region_code}/general_PLOTS')

# Load Validation data 

## Existing VRE sites

In [ ]:
existing_VREs_data_path=Path(f"../data/downloaded_data/CODERS/data-pull/supply/{region_code}_wind_generators.csv")
if existing_VREs_data_path.exists():
    existing_VREs=pd.read_csv(existing_VREs_data_path)
    existing_VREs_gdf=gpd.GeoDataFrame(existing_VREs,geometry=gpd.points_from_xy(existing_VREs.longitude,existing_VREs.latitude),crs=CRS_d)
    utils.print_update(level=1,message=f"Validation data for existing VREs loaded from {existing_VREs_data_path}")
    
    existing_tech_name_mapping={
    'wind_ons':'Wind',
    'solar':'Solar'
    }
    # Create new column 'Technology' based on mapping
    existing_VREs_gdf["Technology"] = existing_VREs_gdf["gen_type"].map(existing_tech_name_mapping)

    # If some gen_type values are not in the dict, fill them with 'Unknown'
    existing_VREs_gdf["Technology"] = existing_VREs_gdf["Technology"].fillna("Unknown")

else:
    existing_VREs_gdf=None
    utils.print_warning(f"Validation data for existing VREs not found at {existing_VREs_data_path}")


if existing_VREs_gdf.crs != CRS_m:
    existing_VREs_plot = existing_VREs_gdf.to_crs(CRS_m)
else:
    existing_VREs_plot = existing_VREs_gdf

## Committed VRE Sites (BCH CFPs)

In [ ]:
committed_VREs_data_path=Path("../ROD_2024/BCH_CFP24.geojson")
if committed_VREs_data_path.exists():
    committed_VREs_gdf=gpd.read_file(committed_VREs_data_path, engine="fiona")
    if committed_VREs_gdf.crs is None:
        committed_VREs_gdf.set_crs(CRS_d, allow_override=True, inplace=True)
    # if committed_VREs_gdf.crs != CRS_m:
    #     committed_VREs_gdf.to_crs(CRS_m, inplace=True)
    utils.print_update(level=1,message=f"Committed VREs data loaded from {committed_VREs_data_path}")
    
    
    committed_tech_name_mapping={
        'wind':'Wind',
        'solar':'Solar'
    }
    # Create new column 'Technology' based on mapping
    committed_VREs_gdf["Technology"] = committed_VREs_gdf["resource_type"].map(existing_tech_name_mapping)

    # If some gen_type values are not in the dict, fill them with 'Unknown'
    committed_VREs_gdf["Technology"] = committed_VREs_gdf["resource_type"].fillna("Unknown")


else:
    committed_VREs_gdf=None
    utils.print_warning(f"Validation data for Committed VREs not found at {committed_VREs_data_path}")


if committed_VREs_gdf.crs != CRS_m:
    committed_VREs_plot = committed_VREs_gdf.to_crs(CRS_m)
else:
    committed_VREs_plot = committed_VREs_gdf

# Load Data from Store

## Load Store

In [ ]:
store_all_runs={}
for POLICY in POLICYs:
    store=f"../data/store/resources_{country_kwd}_{region_code}_{POLICY}.h5"# f"../data/store/resources_{province_code}.h5" 
    res_data=DataHandler(store,show_structure=False) # the DataHandler object could be initiated without the store definition as well.
    store_all_runs[POLICY]=res_data

utils.print_update(level=1,message=f"Loaded store for {len(store_all_runs)} runs for {region_code}:")
utils.print_update(level=2,message="\n    ".join(store_all_runs.keys()))

- If Lines was not used/pulled 

In [ ]:
# lines=gpd.read_file('../data/downloaded_data/OSM/BC_power.geojson')
# # Select columns using boolean indexing
# columns_to_keep = ['power', 'voltage', 'geometry']
# lines_subset = lines.loc[:, lines.columns.isin(columns_to_keep)].copy()

# lines_subset["power"] = lines_subset["power"].astype(str)
# # Handle voltage conversion with error handling for non-numeric values
# lines_subset["voltage"] = pd.to_numeric(lines_subset["voltage"], errors='coerce').fillna(0).astype(int)

# res_data.to_store(lines_subset, 'lines',force_update=True)

- load dfs

In [ ]:
BC_dfs_all_runs={}

for POLICY, res_data in store_all_runs.items():
    utils.print_update(level=1,message=f"Loading dataframes for POLICY: {POLICY}")
    
    # Initialize dictionary for this POLICY
    BC_dfs_all_runs[POLICY] = {}
    BC_dfs_policy = BC_dfs_all_runs[POLICY]
    
    # Loading dataframes
    BC_dfs_policy['cells'] = res_data.from_store('cells')
    BC_dfs_policy['boundary'] = res_data.from_store('boundary')
    BC_dfs_policy['lines'] = res_data.from_store('lines')
    BC_dfs_policy['substations'] = res_data.from_store('substations')
    BC_dfs_policy['timeseries_solar'] = res_data.from_store('timeseries/solar')
    BC_dfs_policy['timeseries_wind'] = res_data.from_store('timeseries/wind')
    BC_dfs_policy['timeseries_clusters_solar'] = res_data.from_store('timeseries/clusters/solar')
    BC_dfs_policy['timeseries_clusters_wind'] = res_data.from_store('timeseries/clusters/wind')
    BC_dfs_policy['clusters_solar'] = res_data.from_store('clusters/solar')
    BC_dfs_policy['clusters_wind'] = res_data.from_store('clusters/wind')

In [ ]:
clusters_wind_BASELINE=BC_dfs_all_runs['BASELINE']['clusters_wind']
clusters_solar_BASELINE=BC_dfs_all_runs['BASELINE']['clusters_solar']
clusters_ts_wind_BASELINE=BC_dfs_all_runs['BASELINE']['timeseries_clusters_wind']
clusters_ts_solar_BASELINE=BC_dfs_all_runs['BASELINE']['timeseries_clusters_solar']

In [ ]:
import RES.RESources as RES_module
top_sites_save_to=f'results/{country_kwd}/{region_code}/TOP_sites'

resource_clusters_solar,cluster_timeseries_solar=RES_module.select_top_sites(clusters_solar_BASELINE,
                                                                clusters_ts_solar_BASELINE,
                                                                    resource_max_capacity=10)
RES_module.export_results(
    resource_type='solar',
    region=region_code,
    resource_clusters=resource_clusters_solar,
    cluster_timeseries=cluster_timeseries_solar,
    save_to=top_sites_save_to,
)
sites_summary:str=RES_module.create_summary_info('solar',
                                    region_code,
                                    resource_clusters_solar,
                                    cluster_timeseries_solar)
RES_module.dump_export_metadata(sites_summary,
                                top_sites_save_to)

resource_clusters_wind,cluster_timeseries_wind=RES_module.select_top_sites(clusters_wind_BASELINE,
                                                                clusters_ts_wind_BASELINE,
                                                                    resource_max_capacity=50)
RES_module.export_results(
    resource_type='wind',
    region=region_code,
    resource_clusters=resource_clusters_wind,
    cluster_timeseries=cluster_timeseries_wind,
    save_to=top_sites_save_to,
)
sites_summary:str=RES_module.create_summary_info('wind',
                                    region_code,
                                    resource_clusters_wind,
                                    cluster_timeseries_wind)
RES_module.dump_export_metadata(sites_summary,
                                top_sites_save_to)

## Extract the Boundary from Store

- creating both degrees and meters projection

In [ ]:
boundary = BC_dfs_policy['boundary']

region_mapping_data_path=Path(f'../data/region_mapping_{region_code}.csv')

if 'Region_number' not in boundary.columns:
    if region_mapping_data_path.exists():
        region_mapping=pd.read_csv(region_mapping_data_path)
        region_mapping['Region_Number'] = range(1, len(region_mapping) + 1)
     
        # Merge region names into boundary using region_mapping
        boundary = boundary.merge(region_mapping, left_on='Region', right_on='Region', how='left')
        # If any Region_Number is NaN, assign sequential numbers
    else:
        boundary['Region_Number'] = range(1, len(boundary) + 1)
        region_mapping = pd.DataFrame({
            'Region': boundary['Region'],
            'Region_Number': boundary['Region_Number']
        })
        
if boundary.crs is None:
    boundary.set_crs(CRS_d, allow_override=True, inplace=True)
if boundary.crs != CRS_m:
    boundary_plot=boundary.to_crs(CRS_m)

- Extracting bounds for clipping purposes

In [ ]:
boundary_aggr=boundary.dissolve()

# Get total bounds from boundary GeoDataFrame
minx, miny, maxx, maxy = boundary.total_bounds

# Create bounding_box_dict with correct keys for downstream use
bounding_box_dict = {
    "minx": float(minx),
    "miny": float(miny),
    "maxx": float(maxx),
    "maxy": float(maxy)
}

# Compare Regional Capacity with Default Scenario

> Run the default scenario results first to set the benchmark

In [ ]:
utils.print_update(level=1,message=" Available run ids: ")
utils.print_update(level=2,message="\n    ".join(BC_dfs_all_runs.keys()))

- Define POLICY from the available results

In [ ]:
POLICY='strict_policy_aeroway_CPCAD_buffer'
BASELINE_results_save_to = utils.ensure_path(f'../results/{country_kwd}/{region_code}/BASELINE/')
POLICY_results_save_to = utils.ensure_path(f'../results/{country_kwd}/{region_code}/{POLICY}/')
BASELINE_vis_save_to = utils.ensure_path(f'../vis/{country_kwd}/{region_code}/BASELINE/')
POLICY_vis_save_to = utils.ensure_path(f'../vis/{country_kwd}/{region_code}/{POLICY}/')

In [ ]:
cells_scenario:pd.DataFrame=BC_dfs_all_runs[f'{POLICY}']['cells']
cells_baseline:pd.DataFrame=BC_dfs_all_runs['BASELINE']['cells']

if cells_scenario.crs is None:
    cells_scenario.set_crs(CRS_d, allow_override=True, inplace=True)
if cells_scenario.crs != CRS_m:
    cells_scenario_proj=cells_scenario.to_crs(CRS_m)

if cells_baseline.crs is None:  
    cells_baseline.set_crs(CRS_d, allow_override=True, inplace=True)
if cells_baseline.crs != CRS_m:
    cells_baseline_proj=cells_baseline.to_crs(CRS_m)
    
cells_scenario.to_csv(POLICY_results_save_to / 'cells_scenario.csv', index=False)
cells_baseline.to_csv(BASELINE_results_save_to / 'cells_baseline.csv', index=False)

---
Temp
---

In [ ]:
clusters_baseline_solar:pd.DataFrame=BC_dfs_all_runs['BASELINE']['clusters_solar']
clusters_baseline_wind:pd.DataFrame=BC_dfs_all_runs['BASELINE']['clusters_wind']

clusters_scenario_solar:pd.DataFrame=BC_dfs_all_runs[f'{POLICY}']['clusters_solar']
clusters_scenario_wind:pd.DataFrame=BC_dfs_all_runs[f'{POLICY}']['clusters_wind']

In [ ]:
clusters_scenario_wind.to_csv(POLICY_results_save_to/f'BC_clusters_scenario_wind_{POLICY}.csv',index=False)
clusters_scenario_solar.to_csv(POLICY_results_save_to/f'BC_clusters_scenario_solar_{POLICY}.csv',index=False)

In [ ]:
# load your df (assuming it's named res_df)
res_df = clusters_scenario_wind.copy()

# mapping to BC Hydro macro-regions
region_map = {
    "PeaceRiver": "Peace",
    "NorthernRockies": "Peace",
    "Kitimat-Stikine": "North Coast",
    "Skeena-QueenCharlotte": "North Coast",
    "EastKootenay": "Southern Interior",
    "CentralOkanagan": "Southern Interior",
    "KootenayBoundary": "Southern Interior",
    "FraserValley": "Southern Interior",
    "SunshineCoast": "Vancouver Island",
    "Nanaimo": "Vancouver Island",
    "ComoxValley": "Vancouver Island",
    "CowichanValley": "Vancouver Island",
    "Strathcona": "Vancouver Island",
}

res_df["BCHydroRegion"] = res_df["Region"].map(region_map)

# aggregate techno-economic indicators
res_summary = (
    res_df.groupby("BCHydroRegion")
    .agg(
        clusters=("Region", "count"),
        total_potential_MW=("potential_capacity", "sum"),
        mean_CF=("CF_mean", "mean"),
        mean_LCOE=("lcoe", "mean"),
    )
    .reset_index()
)

res_summary


In [ ]:
# bc_df = pd.read_csv("../data/validation_data/BC/extracts/BC_onshore_wind_sites.csv")
# bc_df["BCHydroRegion"] = bc_df["region"].replace({
#     "Peace": "Peace",
#     "North Coast": "North Coast",
#     "Southern Interior": "Southern Interior",
#     "Vancouver Island": "Vancouver Island",
#     "Provincewide": "Province-Wide"
# })

# bc_summary = (
#     bc_df.groupby("BCHydroRegion")
#     .agg(
#         official_sites=("option_id", "count"),
#         official_capacity_MW=("installed_capacity_MW", "sum"),
#         official_unit_cost_MWh=("unit_energy_cost_$/MWh", "mean")
#     )
#     .reset_index()
# )

# bc_summary

In [ ]:
# comparison = pd.merge(
#     bc_summary, res_summary, on="BCHydroRegion", how="outer"
# )

# # compute relative indicators
# comparison["Potential/Official_Capacity_Ratio"] = (
#     comparison["total_potential_MW"] / comparison["official_capacity_MW"]
# )

# comparison["LCOE_vs_OfficialRatio"] = (
#     comparison["mean_LCOE"] / comparison["official_unit_cost_MWh"]
# )

# comparison.round(2)


In [ ]:
# comparison["official_capacity_GW"] = comparison["official_capacity_MW"] / 1000
# comparison["potential_capacity_GW"] = comparison["total_potential_MW"] / 1000
# comparison = comparison.sort_values("potential_capacity_GW", ascending=False)


# fig, ax = plt.subplots(figsize=(8, 5))

# comparison.plot(
#     kind="bar",
#     x="BCHydroRegion",
#     y=["official_capacity_GW", "potential_capacity_GW"],
#     ax=ax,
#     color=["#052d55", "#2e95d0"],
#     edgecolor="black"
# )

# ax.set_ylabel("Capacity (GW)")
# ax.set_xlabel("")
# ax.set_title("Onshore Wind Capacity by Region — BC Hydro (2013) vs RESource Potential", fontsize=12)
# ax.legend(["BC Hydro Official (2013)", "RESource Potential (Derived)"], loc="upper right")

# # Format y-axis in GW with one decimal place
# ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.1f}"))

# plt.tight_layout()
# plt.show()

----

- Calculate Aggregated capacity to Regions

In [ ]:
from RES.CellCapacityProcessor import get_sub_nationally_aggregated_capacity

file_path_BASELINE=Path(f'{BASELINE_results_save_to}/cells_aggregated_by_{sub_national_unit_tag}_{region_code}_BASELINE.csv')
file_path_scenario=Path(f'{POLICY_results_save_to}/cells_aggregated_by_{sub_national_unit_tag}_{region_code}_{POLICY}.csv')


utils.print_update(level=2,
                        message=f"Calculating aggregated capacity for {sub_national_unit_tag} | BASELINE...")

# calculate for BASELINE  
cells_aggrs_region_BASELINE=get_sub_nationally_aggregated_capacity(cells_baseline,sub_national_unit_tag)
cells_aggrs_region_BASELINE.to_csv(file_path_BASELINE)
            
utils.print_update(level=3,
                        message=f"Aggregated cells by'{sub_national_unit_tag}' saved to '{file_path_BASELINE}'")


utils.print_update(level=2,
                        message=f"Calculating aggregated capacity for {sub_national_unit_tag} | {POLICY}...")
# Calculate for POLICY
cells_aggrs_region_scenario=get_sub_nationally_aggregated_capacity(cells_scenario,sub_national_unit_tag)
cells_aggrs_region_scenario.to_csv(file_path_scenario)

        
utils.print_update(level=3,
                        message=f"Aggregated cells by'{sub_national_unit_tag}' saved to '{file_path_scenario}'")

- Calculate Capacity variations

In [ ]:
cells_aggrs_region_BASELINE

In [ ]:
delta_cells_aggrs_region=abs(cells_aggrs_region_BASELINE-cells_aggrs_region_scenario)

if delta_cells_aggrs_region.empty:
    utils.print_warning("No differences in aggregated capacities between BASELINE and POLICY scenarios.")
else:
    utils.print_update(level=1,message=f"Differences in aggregated capacities between BASELINE and {POLICY} scenarios:")
    display(delta_cells_aggrs_region.sum()/1E3,"in GW")
    
delta_cells_aggrs_region['percent_change_solar']=round((cells_aggrs_region_BASELINE['potential_capacity_solar'] - cells_aggrs_region_scenario['potential_capacity_solar'])/cells_aggrs_region_BASELINE['potential_capacity_solar']*100,2)
delta_cells_aggrs_region['percent_change_wind']=round((cells_aggrs_region_BASELINE['potential_capacity_wind'] - cells_aggrs_region_scenario['potential_capacity_wind'])/cells_aggrs_region_BASELINE['potential_capacity_wind']*100,2)


In [ ]:
delta_cells_aggrs_region

- Spatial- mapping the delta of capacity to Regions

In [ ]:
if (delta_cells_aggrs_region.sum(axis=0) == 0).all():
    utils.print_update(level=2,
                      message=f"No differences found between default and scenario aggregated cells for {sub_national_unit_tag}.")
    utils.print_warning("Perhaps we are looking at the same scenario?")
else:
    delta_cells_aggrs_region_gdf=boundary.copy()
    
    # - For plotting, Update Boundary (gdf) with this delta_Capacity
    delta_cells_aggrs_region_gdf["potential_capacity_solar"] = delta_cells_aggrs_region_gdf['Region'].str.replace(' ', '').map(delta_cells_aggrs_region["potential_capacity_solar"])
    delta_cells_aggrs_region_gdf["potential_capacity_wind"] = delta_cells_aggrs_region_gdf['Region'].str.replace(' ', '').map(delta_cells_aggrs_region["potential_capacity_wind"])

    # Map Population and GDP to gdf using Region
    delta_cells_aggrs_region_gdf["potential_capacity_solar_GW"] = delta_cells_aggrs_region_gdf["potential_capacity_solar"].apply(lambda x: x / 1E3) # GW
    delta_cells_aggrs_region_gdf["potential_capacity_wind_GW"] = delta_cells_aggrs_region_gdf["potential_capacity_wind"].apply(lambda x: x / 1E3) # GW

- plot in map

In [ ]:
# Plot configuration 
plot_config = { 'solar': {'column': 'potential_capacity_solar_GW', 'cmap': 'YlOrRd', 'label': 'Solar potential (GW)'}, 'wind': {'column': 'potential_capacity_wind_GW', 'cmap': 'BuPu', 'label': 'Wind potential (GW)'} }

In [ ]:
# --- Helper: Plot with shadow + top 5 highlights ---
def plot_map_with_shadow(ax, delta_capacity_gdf, column, cmap, shadow_offset=0.0001, top_n=12):
    """Plot map with shadow effect and highlight top-N rows."""
    
    # Ensure CRS consistency
    if delta_capacity_gdf.crs != CRS_m:
        delta_capacity_gdf = delta_capacity_gdf.to_crs(CRS_m)

    # Shadow
    shadow_geom = delta_capacity_gdf.geometry.translate(xoff=shadow_offset, yoff=-shadow_offset)
    gpd.GeoDataFrame(geometry=shadow_geom, crs=CRS_m).plot(
        ax=ax, facecolor='none', edgecolor='gray', linewidth=1.3, alpha=0.3
    )

    # Main plot
    delta_capacity_gdf.plot(column=column, ax=ax, cmap=cmap, edgecolor='black', linewidth=0.2, legend=False)

    # --- Highlight top N values ---
    top_gdf = delta_capacity_gdf.nlargest(top_n, column)
    for _, row in top_gdf.iterrows():
        if row[column] > 0:
            x, y = row.geometry.centroid.x, row.geometry.centroid.y
            ax.text(x, y, f"{row[column]:.1f}",
                    ha="center", va="center", fontsize=10, fontweight="bold", color="white",
                    path_effects=[pe.withStroke(linewidth=1.1, foreground="k", alpha=0.6)])

     # region label (slightly above the numeric one)
            region_name = str(row.get("Region", ""))
            if region_name:
                ax.text(x, y + 35000, region_name,  # adjust offset depending on projection scale
                        ha="center", va="bottom", fontsize=7, fontweight="regular",
                        color="black",
                        path_effects=[
                            pe.withStroke(linewidth=1.5, foreground="white", alpha=0.6)
                        ])
    
    
    # Return scalar mappable for colorbar
    return plt.cm.ScalarMappable(
        cmap=cmap,
        norm=plt.Normalize(vmin=delta_capacity_gdf[column].min(),
                           vmax=delta_capacity_gdf[column].max())
    )

# --- Create plots ---
fig, axes = plt.subplots(dpi=1000, ncols=2, figsize=(8, 3),constrained_layout=True)
fig.suptitle("Potential capacity lost due to landuse policy implication",
             weight='bold', fontsize=12,y=1)
plt.subplots_adjust(wspace=0.001)  # reduce horizontal space between subplots
for ax in axes:
    ax.set_axis_off()

for ax, (key, config) in zip(axes, plot_config.items()):
    sm = plot_map_with_shadow(ax, delta_cells_aggrs_region_gdf,
                              config['column'], config['cmap'], top_n=8)
    cbar = fig.colorbar(sm, ax=ax, shrink=0.6)
    cbar.set_label(config['label'], fontsize=12)

# plt.tight_layout()
plt.savefig(f'{POLICY_vis_save_to}/potential_capacity_lost_default_vs_{POLICY}.svg',
            bbox_inches='tight')

#DOC content
# plt.savefig('../docs/source/_static/potential_capacity_lost_default_vs_policy_aeroway_CPCAD_buffer.png')


In [ ]:
policy='strict_policy_aeroway_CPCAD_buffer'


In [ ]:
cells= BC_dfs_all_runs[policy]['cells']

if cells.crs!=CRS_m:
    cells_plot= cells.to_crs(CRS_m)
else:
    cells_plot=cells

In [ ]:
cells_clean_solar = cells_plot[(cells_plot['potential_capacity_solar'] >= 1) &(cells_plot['solar_CF_mean'] > 0)]
cells_clean_wind = cells_plot[(cells_plot['potential_capacity_wind'] >= 3) &(cells_plot['wind_CF_mean'] > 0)]

#### Lost capacity and Score

In [ ]:
score_threshold=200 # $/MWh

In [ ]:
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
import matplotlib.pyplot as plt

# =============================
# 1️⃣ Create Figure and Subplots
# =============================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5), dpi=1000)
# fig.suptitle(f"Resources and Capacity by Country – {country_name}", fontsize=18, fontweight='bold')

# =============================
# 2️⃣ Plot resource maps (from first block)
# =============================
vis.get_data_in_map_plot(
    cells_clean_solar, 
    resource_type='solar',
    datafield='score',
    compass_size=12,
    ax=ax1, 
    score_threshold=250,
    show=False
)

vis.get_data_in_map_plot(
    cells_clean_wind, 
    resource_type='wind',
    datafield='score',
    ax=ax2, 
    score_threshold=250,
    show=False
)

# =============================
# 3️⃣ Overlay “Cells without suitable land”
# =============================
cells_plot.plot(ax=ax1, color='grey', alpha=1, zorder=1)
cells_plot.plot(ax=ax2, color='grey', alpha=1, zorder=1)

# Create legend patch
no_land_patch = mpatches.Patch(
    facecolor='grey',
    edgecolor='lightgray',
    alpha=0.5,
    label=f'Cells crossing score threshold {score_threshold} $/MWh or lacks suitable land'
)

# =============================
# 4️⃣ Add existing solar/wind projects
# =============================
existing_VREs_gdf_solar = existing_VREs_plot[existing_VREs_plot['Technology'].str.lower() == 'solar']
committed_VREs_plot_solar = committed_VREs_plot[committed_VREs_plot['Technology'].str.lower() == 'solar']
ax1, solar_legends = vis.get_existing_committed_VRE_plot(
    ax=ax1,
    existing_VREs_gdf=existing_VREs_gdf_solar,
    existing_VRE_type_column='Technology',
    committed_VREs_gdf=committed_VREs_plot_solar,
    committed_VRE_type_column='Technology',
    target_crs=CRS_m,
    marker_scale_existing=0.4,
    marker_scale_committed=0.4,
    marker_highlight_width=4
)

existing_VREs_gdf_wind = existing_VREs_plot[existing_VREs_plot['Technology'].str.lower() == 'wind']
committed_VREs_plot_wind = committed_VREs_plot[committed_VREs_plot['Technology'].str.lower() == 'wind']
ax2, wind_legends = vis.get_existing_committed_VRE_plot(
    ax=ax2,
    existing_VREs_gdf=existing_VREs_gdf_wind,
    existing_VRE_type_column='Technology',
    committed_VREs_gdf=committed_VREs_plot_wind,
    committed_VRE_type_column='Technology',
    target_crs=CRS_m,
    marker_scale_existing=0.2,
    marker_scale_committed=0.2,
    marker_highlight_width=4
)

# =============================
# 5️⃣ Overlay country boundaries and labels (from second block)
# =============================
Country_name_Y_adjustment = 22E3  # meters in CRS_m
cells_aggr_plot=delta_cells_aggrs_region_gdf.to_crs(CRS_m)
fig.suptitle(f"Capacity Lost due to land-use policy changes and their relative scores – {country_name}", fontsize=18, fontweight='bold', y=1.05)

# Plot only boundaries (no fill)
boundary_plot.plot(ax=ax1, facecolor='none', edgecolor='black', linewidth=0.6, zorder=3)
boundary_plot.plot(ax=ax2, facecolor='none', edgecolor='black', linewidth=0.6, zorder=3)

# --- Pick top 5 by solar potential ---
top5_solar = (
    cells_aggr_plot
    .sort_values(by='potential_capacity_solar_GW', ascending=False)
    .head(5)
)
# Add text annotations for capacity and country name
for idx, row in top5_solar.iterrows():
    centroid = row.geometry.centroid

    # Solar capacity on left
    ax1.annotate(
        f"{row['potential_capacity_solar_GW']:.1f}",
        (centroid.x, centroid.y),
        color="black",
        fontsize=10,
        ha="center",
        va="center",
        fontweight="bold",
        zorder=4,
        path_effects=[pe.withStroke(linewidth=3, foreground="white", alpha=0.9)]
    )
    ax1.annotate(
        row[sub_national_unit_tag],
        (centroid.x, centroid.y + Country_name_Y_adjustment),
        color="black",
        fontsize=9,
        ha="center",
        va="bottom",
        zorder=4,
        path_effects=[pe.withStroke(linewidth=3, foreground="white", alpha=0.6)]
    )

top5_wind = (
    cells_aggr_plot
    .sort_values(by='potential_capacity_wind_GW', ascending=False)
    .head(5)
)
for idx, row in top5_wind.iterrows():
    centroid = row.geometry.centroid
    # Wind capacity on right
    ax2.annotate(
        f"{row['potential_capacity_wind_GW']:.1f}",
        (centroid.x, centroid.y),
        color="black",
        fontsize=10,
        ha="center",
        va="center",
        fontweight="bold",
        zorder=4,
        path_effects=[pe.withStroke(linewidth=3, foreground="white", alpha=0.9)]
    )
    ax2.annotate(
        row[sub_national_unit_tag],
        (centroid.x, centroid.y + Country_name_Y_adjustment),
        color="black",
        fontsize=9,
        ha="center",
        va="bottom",
        zorder=4,
        path_effects=[pe.withStroke(linewidth=3, foreground="white", alpha=0.6)]
    )

# =============================
# Add legends and note
# =============================
# Combine legend handles from both plots
combined_legend_handles = solar_legends + wind_legends + [no_land_patch]

# Extract labels
labels = [h.get_label() for h in combined_legend_handles]

# --- Deduplicate using dictionary (preserves order) ---
unique = dict(zip(labels, combined_legend_handles))
unique_labels = list(unique.keys())
unique_handles = list(unique.values())
wrapped_labels = [textwrap.fill(label, width=30) for label in unique_labels]

fig.legend(
    handles=unique_handles,
    labels=wrapped_labels,
    loc='upper center',
    bbox_to_anchor=(0.45, 0.98),
    ncol=1,
    fontsize=8,
    frameon=False,
    handlelength=1.5,    # horizontal length of the legend handle
    handleheight=1.2,    # vertical spacing
    markerscale=0.7      # scales marker size relative to the plot markers
)

fig.text(
    0.5, -0.05,
    "Note: The Scoring reflects relative investment per MWh yield. Values above 250 $/MWh are considered non-feasible. "
    "Country-level potentials (in GW) are annotated from aggregated site capacities.",
    ha='center',
    va='top',
    fontsize=9,
    color='gray',
    wrap=True,
)

plt.tight_layout()

plt.savefig(common_vis_save_to/"Resources_and_Capacity_lost_policy.png", bbox_inches='tight', transparent=False)

# Maps

* Combined Map

In [ ]:
supported_datafields=['CF','CAPACITY'] # that's what I configured in the plot func

In [ ]:
for datafield in supported_datafields:
    for policy in POLICYs:
        print(f"Generating combined resource plots for datafield: {datafield} | policy: {policy}...")
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10,5), dpi=500)

        cells= BC_dfs_all_runs[policy]['cells']
        if cells.crs!=boundary_plot.crs:
            cells_proj= cells.to_crs(boundary_plot.crs)
        # fig.suptitle(f"Renewable Resources in {region_name} ({policy})", fontsize=16, fontweight='bold')
        vis.get_data_in_map_plot(cells_proj, 
                        resource_type='solar',
                        datafield=datafield,
                        ax=ax1, 
                        score_threshold=600,
                        show=False)
        vis.get_data_in_map_plot(cells_proj, 
                        resource_type='wind',
                        datafield=datafield,
                        ax=ax2, 
                        score_threshold=600,
                        show=False)
        
        # vis.add_compass_arrow_custom(ax1, text_offset=0.04)
        vis.add_compass_arrow_custom(ax2, text_offset=0.04)
        plt.tight_layout()

        
        
        # Add existing VREs to the plot

        existing_VREs_gdf_solar=existing_VREs_plot[existing_VREs_plot['Technology'].str.lower()=='solar']
        ax1,solar_legends=vis.get_existing_committed_VRE_plot(ax=ax1,
                                                            existing_VREs_gdf=existing_VREs_gdf_solar,
                                                            existing_VRE_type_column='Technology',
                                                            committed_VREs_gdf=committed_VREs_plot,
                                                            committed_VRE_type_column='Technology',
                                                            target_crs=CRS_m,
                                                            marker_scale_existing=0.2,
                                                            marker_scale_committed=0.2,
                                                            marker_highlight_width=2)
        
        existing_VREs_gdf_wind=existing_VREs_plot[existing_VREs_plot['Technology'].str.lower()=='wind']
        ax2,wind_legends=vis.get_existing_committed_VRE_plot(ax=ax2,
                                                            existing_VREs_gdf=existing_VREs_gdf_wind,
                                                            existing_VRE_type_column='Technology',
                                                            committed_VREs_gdf=committed_VREs_plot,
                                                            committed_VRE_type_column='Technology',
                                                            target_crs=CRS_m,
                                                            marker_scale_existing=0.2,
                                                            marker_scale_committed=0.2,
                                                            marker_highlight_width=2)
        # Combine legend handles from both plots
        combined_legend_handles = solar_legends + wind_legends

        # Extract labels
        labels = [h.get_label() for h in combined_legend_handles]

        # --- Deduplicate using dictionary (preserves order) ---
        unique = dict(zip(labels, combined_legend_handles))
        unique_labels = list(unique.keys())
        unique_handles = list(unique.values())
        
        fig.legend(
            handles=unique_handles,
            labels=unique_labels,
            loc='upper center',
            bbox_to_anchor=(0.45, 0.88),
            ncol=1,
            fontsize=8,
            frameon=False,
            handlelength=1.5,    # horizontal length of the legend handle
            handleheight=1.2,    # vertical spacing
            markerscale=0.7      # scales marker size relative to the plot markers
        )
        
        save_to_root= BASELINE_vis_save_to if policy=='BASELINE' else POLICY_vis_save_to
                
        plt.savefig(save_to_root/f"Resources_combined_{datafield}_{policy}.png", bbox_inches='tight', transparent=False)
        utils.print_update(level=2,
                        message=f"Combined resource plot for datafield: {datafield} | policy: {policy} saved to {save_to_root}/Resources_combined_{datafield}_{policy}.png")
        plt.show()

### Score

- Prepare Cells

In [ ]:
cells= BC_dfs_all_runs[policy]['cells']

if cells.crs!=CRS_m:
    cells_plot= cells.to_crs(CRS_m)
else:
    cells_plot=cells

In [ ]:
cells_clean_solar = cells_plot[(cells_plot['potential_capacity_solar'] >= 1) &(cells_plot['solar_CF_mean'] > 0)]
cells_clean_wind = cells_plot[(cells_plot['potential_capacity_wind'] >= 3) &(cells_plot['wind_CF_mean'] > 0)]

##### Score with Capacity

In [ ]:
score_threshold_solar=120 # $/MWh
score_threshold_wind=150 # $/MWh

In [ ]:
cells_solar_filtered=cells_clean_solar[cells_clean_solar['lcoe_solar']<=score_threshold_solar]
cells_wind_filtered=cells_clean_wind[cells_clean_wind['lcoe_wind']<=score_threshold_wind]

- Score Mapping (bins)

In [ ]:
# Define custom bins and labels for solar and wind capacity
solar_bins = [30, 40, 50, 60, 70,80, 90,100, float('inf')]  # Custom ranges
solar_labels = ['<30','30-40','40-50','50-60','60-70', '70-80', '80-90','>100']  # Labels for legend
# Define custom bins and labels for solar and wind capacity
wind_bins = [40, 50, 60, 70, 80, 90, 100, 120, float('inf')]
wind_labels = ['40-50','50-60','60-70','70-80','80-90','90-100','100-120','>120']

In [ ]:
# Categorize potential_capacity_solar and potential_capacity_wind into bins
cells_plot_solar=cells_solar_filtered.copy()
cells_plot_solar['solar_category'] = pd.cut(cells_plot_solar['lcoe_solar'], bins=solar_bins, labels=solar_labels, include_lowest=True)

cells_plot_wind=cells_wind_filtered.copy()
cells_plot_wind['wind_category'] = pd.cut(cells_plot_wind['lcoe_wind'], bins=wind_bins, labels=wind_labels, include_lowest=True)

- Prepare Aggregated Capacity for map overlay layer

In [ ]:
cells_aggrs_solar=get_sub_nationally_aggregated_capacity(cells_plot_solar,sub_national_unit_tag)
cells_aggrs_wind=get_sub_nationally_aggregated_capacity(cells_plot_wind,sub_national_unit_tag)

In [ ]:
cells_aggr_solar_plot=boundary_plot.copy()
# - For plotting, Update Boundary (gdf) with this delta_Capacity
cells_aggr_solar_plot["potential_capacity_solar"] = boundary_plot['Region'].str.replace(' ', '').map(cells_aggrs_region_scenario["potential_capacity_solar"])
cells_aggr_solar_plot["potential_capacity_wind"] = boundary_plot['Region'].str.replace(' ', '').map(cells_aggrs_region_scenario["potential_capacity_wind"])

# # Map Population and GDP to gdf using Region
cells_aggr_solar_plot["potential_capacity_solar_GW"] = cells_aggr_solar_plot["potential_capacity_solar"].apply(lambda x: x / 1E3) # GW


In [ ]:
cells_aggr_wind_plot=boundary_plot.copy()
# - For plotting, Update Boundary (gdf) with this delta_Capacity
cells_aggr_wind_plot["potential_capacity_solar"] = boundary_plot['Region'].str.replace(' ', '').map(cells_aggrs_region_scenario["potential_capacity_solar"])
cells_aggr_wind_plot["potential_capacity_wind"] = boundary_plot['Region'].str.replace(' ', '').map(cells_aggrs_region_scenario["potential_capacity_wind"])

# # Map Population and GDP to gdf using Region
cells_aggr_wind_plot["potential_capacity_wind_GW"] = cells_aggr_wind_plot["potential_capacity_wind"].apply(lambda x: x / 1E3) # GW

- Plot

In [ ]:
import textwrap
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
import matplotlib.pyplot as plt

# =============================
# 1️⃣ Create Figure and Subplots
# =============================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 5), dpi=1000)
ax1.set_axis_off()
ax2.set_axis_off()
# fig.suptitle(f"Resources and Capacity by Country – {country_name}", fontsize=18, fontweight='bold')

# =============================
# 2️⃣ Plot resource maps (from first block)
# =============================

# vis.get_data_in_map_plot(
#     cells_plot_solar, 
#     resource_type='solar',
#     datafield='score',
#     compass_size=12,
#     ax=ax1, 
#     score_threshold=score_threshold_solar,
#     show=False
# )

cells_plot_solar.plot(ax=ax1, column='solar_category', cmap='YlOrRd', legend=False, edgecolor='white', linewidth=0.3, zorder=2)

# vis.get_data_in_map_plot(
#     cells_plot_wind, 
#     resource_type='wind',
#     datafield='score',
#     ax=ax2, 
#     score_threshold=score_threshold_wind,
#     show=False
# )
cells_plot_wind.plot(ax=ax2, column='wind_category', cmap='BuPu', legend=False, edgecolor='white', linewidth=0.3, zorder=2)

# =============================
# 3️⃣ Overlay “Cells without suitable land”
# =============================
cells_plot.plot(ax=ax1, color='grey', edgecolor='white', linewidth=0.3, alpha=0.5, zorder=1)
cells_plot.plot(ax=ax2, color='grey', edgecolor='white', linewidth=0.3, alpha=0.5, zorder=1)

# Create legend patch
no_land_patch = mpatches.Patch(
    facecolor='grey',
    edgecolor='lightgray',
    alpha=0.5,
    label=f'Cells crossing score threshold [solar : {score_threshold_solar}, wind : {score_threshold_wind} $/MWh] or lacks suitable land'
)

# =============================
# 4️⃣ Add existing solar/wind projects
# =============================
existing_VREs_gdf_solar = existing_VREs_plot[existing_VREs_plot['Technology'].str.lower() == 'solar']
committed_VREs_plot_solar = committed_VREs_plot[committed_VREs_plot['Technology'].str.lower() == 'solar']
ax1, solar_legends = vis.get_existing_committed_VRE_plot(
    ax=ax1,
    existing_VREs_gdf=existing_VREs_gdf_solar,
    existing_VRE_type_column='Technology',
    committed_VREs_gdf=committed_VREs_plot_solar,
    committed_VRE_type_column='Technology',
    target_crs=CRS_m,
    marker_scale_existing=0.2,
    marker_scale_committed=0.2,
    marker_highlight_width=4
)

existing_VREs_gdf_wind = existing_VREs_plot[existing_VREs_plot['Technology'].str.lower() == 'wind']
committed_VREs_plot_wind = committed_VREs_plot[committed_VREs_plot['Technology'].str.lower() == 'wind']
ax2, wind_legends = vis.get_existing_committed_VRE_plot(
    ax=ax2,
    existing_VREs_gdf=existing_VREs_gdf_wind,
    existing_VRE_type_column='Technology',
    committed_VREs_gdf=committed_VREs_plot_wind,
    committed_VRE_type_column='Technology',
    target_crs=CRS_m,
    marker_scale_existing=0.2,
    marker_scale_committed=0.2,
    marker_highlight_width=4
)

# =============================
# 5️⃣ Overlay country boundaries and labels (from second block)
# =============================
Country_name_Y_adjustment = 22E3  # meters in CRS_m

# fig.suptitle(f"Sub-provincial potentials and their relative scores – {country_name}", fontsize=18, fontweight='bold', y=1.05)

# Plot only boundaries (no fill)
boundary_plot.plot(ax=ax1, facecolor='none', edgecolor='grey', linewidth=0.3, zorder=3)
boundary_plot.plot(ax=ax2, facecolor='none', edgecolor='grey', linewidth=0.3, zorder=3)

# --- Pick top 5 by solar potential ---
top5_solar = (
    cells_aggr_solar_plot
    .sort_values(by='potential_capacity_solar_GW', ascending=False)
    .head(6)
)
# Add text annotations for capacity and country name
for idx, row in top5_solar.iterrows():
    centroid = row.geometry.centroid

    # Solar capacity on left
    ax1.annotate(
        f"{row['potential_capacity_solar_GW']:.1f}",
        (centroid.x, centroid.y),
        color="black",
        fontsize=10,
        ha="center",
        va="center",
        fontweight="bold",
        zorder=4,
        path_effects=[pe.withStroke(linewidth=3, foreground="white", alpha=0.9)]
    )
    ax1.annotate(
        row[sub_national_unit_tag],
        (centroid.x, centroid.y + Country_name_Y_adjustment),
        color="black",
        fontsize=9,
        ha="center",
        va="bottom",
        zorder=4,
        path_effects=[pe.withStroke(linewidth=3, foreground="white", alpha=0.8)]
    )

top5_wind = (
    cells_aggr_wind_plot
    .sort_values(by='potential_capacity_wind_GW', ascending=False)
    .head(7)
)
for idx, row in top5_wind.iterrows():
    centroid = row.geometry.centroid
    # Wind capacity on right
    ax2.annotate(
        f"{row['potential_capacity_wind_GW']:.1f}",
        (centroid.x, centroid.y),
        color="black",
        fontsize=10,
        ha="center",
        va="center",
        fontweight="bold",
        zorder=4,
        path_effects=[pe.withStroke(linewidth=3, foreground="white", alpha=0.9)]
    )
    ax2.annotate(
        row[sub_national_unit_tag],
        (centroid.x, centroid.y + Country_name_Y_adjustment),
        color="black",
        fontsize=9,
        ha="center",
        va="bottom",
        zorder=4,
        path_effects=[pe.withStroke(linewidth=3, foreground="white", alpha=0.8)]
    )

# =============================
# Add legends and note
# =============================
# Combine legend handles from both plots
combined_legend_handles = [no_land_patch] + solar_legends + wind_legends

# Extract labels
labels = [h.get_label() for h in combined_legend_handles]

# --- Deduplicate using dictionary (preserves order) ---
unique = dict(zip(labels, combined_legend_handles))
unique_labels = list(unique.keys())
unique_handles = list(unique.values())
wrapped_labels = [textwrap.fill(label, width=30) for label in unique_labels]

fig.legend(
    handles=unique_handles,
    labels=wrapped_labels,
    loc='upper center',
    bbox_to_anchor=(0.43, 0.8),
    ncol=1,
    fontsize=9,
    frameon=False,
    handlelength=1.6,    # horizontal length of the legend handle
    handleheight=1.2,    # vertical spacing
    markerscale=0.7      # scales marker size relative to the plot markers
)


plt.tight_layout()
plt.savefig(common_vis_save_to/"Resources_and_Capacity_Combined.png", bbox_inches='tight', transparent=True)

In [ ]:
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt

# Define colormaps
solar_cmap = cm.get_cmap('YlOrRd', len(solar_labels))
wind_cmap = cm.get_cmap('BuPu', len(wind_labels))

# Generate colors for each bin
solar_colors = [mcolors.rgb2hex(solar_cmap(i)) for i in range(len(solar_labels))]
wind_colors = [mcolors.rgb2hex(wind_cmap(i)) for i in range(len(wind_labels))]

# Aggregate potential capacity for each bin
solar_capacity = (
    cells_plot_solar.groupby('solar_category')['potential_capacity_solar']
    .sum().div(1e3)
    .reindex(solar_labels, fill_value=0)
)
wind_capacity = (
    cells_plot_wind.groupby('wind_category')['potential_capacity_wind']
    .sum().div(1e3)
    .reindex(wind_labels, fill_value=0)
)

# --- Drop bins with 0 capacity ---
solar_capacity = solar_capacity[solar_capacity > 0]
wind_capacity = wind_capacity[wind_capacity > 0]

solar_colors_filtered = [solar_colors[solar_labels.index(lbl)] for lbl in solar_capacity.index]
wind_colors_filtered = [wind_colors[wind_labels.index(lbl)] for lbl in wind_capacity.index]

In [ ]:
### Solar Plot (Horizontal, lowest cost on top) ###
fig1, ax1 = plt.subplots(figsize=(6, 3), dpi=500, constrained_layout=True)
fig1.patch.set_alpha(0)
ax1.set_facecolor('none')

# Horizontal bar plot
ax1.barh(solar_capacity.index, solar_capacity.values,
         color=solar_colors_filtered, edgecolor='k',linewidth=0.2)

ax1.set_xlabel('Solar Potential (GW)', fontsize=14, weight='bold')
ax1.set_ylabel('Relative cost score ($/MWh)', fontsize=14)

# Put lowest scores at the top
ax1.invert_yaxis()

ax1.grid(False)

for spine in ax1.spines.values():
    spine.set_visible(False)
ax1.tick_params(bottom=True, left=True, labelsize=14)

plt.savefig(f'{common_vis_save_to}/cost_barchart_solar.svg', transparent=True)

In [ ]:
### Wind Plot (Horizontal, lowest cost on top) ###
fig2, ax2 = plt.subplots(figsize=(6, 3), dpi=500, constrained_layout=True)
fig2.patch.set_alpha(0)
ax2.set_facecolor('none')

# Horizontal bar plot
ax2.barh(wind_capacity.index, wind_capacity.values,
         color=wind_colors_filtered, edgecolor='k',linewidth=0.2)

ax2.set_xlabel('Wind Potential (GW)', fontsize=14, weight='bold')
ax2.set_ylabel('Relative cost score ($/MWh)', fontsize=14)

# Put lowest scores at the top
ax2.invert_yaxis()

ax2.grid(False)

for spine in ax2.spines.values():
    spine.set_visible(False)
ax2.tick_params(bottom=True, left=True, labelsize=14)

plt.savefig(f'{common_vis_save_to}/cost_barchart_wind.svg', transparent=True)

## CF checks

In [ ]:
gwa_country_code=cfg_policy.get('region_mapping').get(region_code).get('GWA_country_code')
utils.print_banner(f"GWA Country Code Selected: {gwa_country_code}")

### Wind

In [ ]:
# Get total bounds from boundary GeoDataFrame
minx, miny, maxx, maxy = boundary.total_bounds

# Create bounding_box_dict with correct keys for downstream use
bounding_box_dict = {
    "minx": float(minx),
    "miny": float(miny),
    "maxx": float(maxx),
    "maxy": float(maxy)
}


In [ ]:
import rioxarray as rxr

raster_path=f'../data/downloaded_data/GWA/{gwa_country_code}_capacity-factor_IEC3.tif'
gwa_raster_data = (
        rxr.open_rasterio(raster_path)
        .rio.clip_box(**bounding_box_dict)
        .rename('CF_IEC3')
        .drop_vars(['band', 'spatial_ref'])
        .isel(band=1 if '*Class*' in 'CF_IEC3' else 0)  # 'IEC_Class_ExLoads' data is in band 1
    )


In [ ]:
# import matplotlib.pyplot as plt

# fig, ax = plt.subplots(figsize=(3.5, 2.5),dpi=500)
# gwa_raster_data.plot(ax=ax, cmap='BuPu', add_colorbar=True)
# boundary.plot(ax=ax, facecolor='none', edgecolor='white', linewidth=0.5)
# ax.set_title("GWA CF-IEC3 Reference (High-res)")
# ax.axis('off')
# plt.savefig(f"../vis/{region_code}/GWA_CF_IEC3.png", bbox_inches='tight', transparent=False)

In [ ]:
# vis.get_CF_wind_check_plot(cells, 
#                        gwa_raster_data,
#                        boundary,
#                        region_code,
#                        region_name,
#                        ['CF_IEC3', 'wind_CF_mean'],
#                        figure_height=7,)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set a clean and minimal style
sns.set_style("white")

# Initialize the figure
plt.figure(figsize=(7.5, 2.5),dpi=1000)
plt.style.use('../RES/visual_styles/elsevier.mplstyle')

# Create the boxplot
ax = sns.boxplot(
    data=cells[['CF_IEC2', 'CF_IEC3', 'wind_CF_mean']],
    palette="Paired",
    linewidth=0.2,
    width=0.6
)

# Set title and labels
ax.set_title('Capacity Factor Distribution Comparison', weight='semibold', pad=12)
ax.set_ylabel('Capacity Factor')
ax.set_xlabel('')

# Tweak tick formatting
ax.tick_params(axis='x')
ax.tick_params(axis='y')

# Remove all spines
for spine in ax.spines.values():
    spine.set_visible(False)

# Add horizontal grid lines
ax.yaxis.grid(True, linestyle='--', alpha=0.4)
ax.xaxis.grid(False)

plt.figtext(
    0.01, -0.08,
    "*'CF_IEC2, CF_IEC3: Average CF for ERA5 Cells, calculated from high-resolution yearly average CF from GWA for IEC Class 2 and 3 turbines.\n"
    "*wind_CF_mean: Average CF for ERA5 Cells, calculated from the ERA5 windspeed (rescaled with GWA) time series ",
    ha='left', fontsize=7, style='normal', fontweight='normal', color='gray',wrap=True
)

plt.tight_layout()
plt.savefig(f"../vis/{country_kwd}/{region_code}/CF_distribution_comparison.svg", bbox_inches='tight', transparent=False)

# Clusters

In [ ]:
clusters_wind=BC_dfs_all_runs[f'{POLICY}']['clusters_wind']
clusters_solar=BC_dfs_all_runs[f'{POLICY}']['clusters_solar']

In [ ]:
clusters_wind_f=clusters_wind[clusters_wind['potential_capacity']>0]
clusters_solar_f=clusters_solar[clusters_solar['potential_capacity']>0]

In [ ]:
print(f'Total sites {len(clusters_wind_f)}')
total_capacity=clusters_wind_f.potential_capacity.sum()
print(f'Total Capacity {int(total_capacity/1E3)} GW')
sites=5
top_sites_capacity=clusters_wind_f.head(sites).potential_capacity.sum()
print(f'Top {sites} sites ({round(sites/len(clusters_wind_f)*100)}% site) capacity {int(top_sites_capacity/1E3)} GW ({round(top_sites_capacity/total_capacity*100)}% of total capacity)')

In [ ]:
print(f'Total sites {len(clusters_solar_f)}')
total_capacity=clusters_solar_f.potential_capacity.sum()
print(f'Total Capacity {int(total_capacity/1E3)} GW')
sites=5
top_sites_capacity=clusters_solar_f.head(sites).potential_capacity.sum()
print(f'Top {sites} sites ({round(sites/len(clusters_solar_f)*100)}% site) capacity {int(top_sites_capacity/1E3)} GW ({round(top_sites_capacity/total_capacity*100)}% of total capacity)')

In [ ]:
clusters_wind_f=clusters_wind[clusters_wind['potential_capacity']>0]
clusters_solar_f=clusters_solar[clusters_solar['potential_capacity']>0]

## Cluster-Timeseries


In [ ]:
dissolved_indices_solar=store_all_runs[f"{POLICY}"].from_store('dissolved_indices/solar')
cell_ts_solar=store_all_runs[f"{POLICY}"].from_store('timeseries/solar')
dissolved_indices_wind=store_all_runs[f"{POLICY}"].from_store('dissolved_indices/wind')
cell_ts_wind=store_all_runs[f"{POLICY}"].from_store('timeseries/wind')

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def get_cluster_vs_cells_profile(region_to_plot: str, 
                                 cluster_id_to_plot: int,
                                 resource_type:str,
                                 cells_timeseries: pd.DataFrame, 
                                 dissolved_indices: pd.DataFrame, 
                                 cluster_timeseries: pd.DataFrame,
                                 plot_save_to: str | Path = None):
    """
    Plot cluster representative profile vs all member cells as daily mean ± std deviation.

    Parameters
    ----------
    region_to_plot : str
        Region name.
    cluster_id_to_plot : int
        Cluster ID.
    cells_timeseries : pd.DataFrame
        Timeseries for all cells, indexed by datetime.
    dissolved_indices : pd.DataFrame
        Mapping of cells to clusters: dissolved_indices.loc[region, cluster_id] gives list of cell IDs.
    cluster_timeseries : pd.DataFrame
        Cluster representative timeseries, indexed by datetime.
    plot_save_to : str | Path, optional
        Path to save figure. If None, figure is saved under `./vis/`.
    """
    resource_type = resource_type.lower()
    if resource_type not in ['solar', 'wind']:
        raise ValueError("resource_type must be either 'solar' or 'wind'")
    region = region_to_plot
    cluster_id = cluster_id_to_plot

    # --- 1. GET MEMBER CELLS OF THE CLUSTER ---
    cell_ids = dissolved_indices.loc[region, cluster_id]

    # --- 2. RESAMPLE TO DAILY MEAN ---
    cell_daily = [cells_timeseries[cid].resample("1D").mean() for cid in cell_ids]
    cluster_daily = cluster_timeseries[f"{region}_{cluster_id}"].resample("1D").mean()

    # --- 3. CONVERT TO 2D ARRAY (days × cells) ---
    cell_matrix = np.column_stack([s.values for s in cell_daily])

    # --- 4. CALCULATE DAILY MEAN AND STD DEV ---
    mean_cells = cell_matrix.mean(axis=1)
    std_cells = cell_matrix.std(axis=1)

    # --- 5. PLOT ---
    fig, ax = plt.subplots(figsize=(12, 3.5), dpi=1000)

    # Shaded area: ±1 std deviation
    ax.fill_between(cell_daily[0].index, mean_cells - std_cells, mean_cells + std_cells,
                    color="orange" if resource_type=='solar' else "skyblue", alpha=0.3, label="Cells ±1 Std Dev")

    # Cluster representative
    ax.plot(cluster_daily.index, cluster_daily.values, color="orangered" if resource_type=='solar' else "navy", linewidth=2, label="Cluster Profile")

    # Clean aesthetics
    ax.set_title(f"Daily Mean {resource_type.capitalize()} Profiles – {region} Cluster {cluster_id}", fontsize=14, weight="bold")
    # ax.set_xlabel("Day of Year", fontsize=12)
    ax.set_ylabel("Normalized Generation", fontsize=12)
    ax.grid(alpha=0.4,linestyle='--', linewidth=0.5)
    ax.legend(frameon=False,fontsize=13)
    ax.tick_params(axis="both", which="major", labelsize=10,labelrotation =90,direction='in',length=3)

    # Optional: make background transparent
    # fig.patch.set_alpha(0)
    # ax.set_facecolor('none')

    plt.tight_layout()

    # --- 6. SAVE FIGURE ---
    plot_save_to = Path(f'{POLICY_vis_save_to}/{resource_type}_cluster_{region}_{cluster_id}_vs_cells_profile.svg')
    plot_save_to.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(plot_save_to)
    utils.print_update(level=2, message=f"Cluster vs Cells profile plot saved to {plot_save_to}")
    plt.show()

In [ ]:
timeseries_clusters_solar=BC_dfs_all_runs[f'{POLICY}']['timeseries_clusters_solar']
timeseries_clusters_wind=BC_dfs_all_runs[f'{POLICY}']['timeseries_clusters_wind']

In [ ]:
get_cluster_vs_cells_profile('EastKootenay', 1, 'solar',cell_ts_solar, dissolved_indices_solar, timeseries_clusters_solar)
get_cluster_vs_cells_profile('PeaceRiver', 1, 'wind',cell_ts_wind, dissolved_indices_wind, timeseries_clusters_wind)

In [ ]:
# resource_clusters_solar,cluster_timeseries_solar=Builder.select_top_sites(solar_clusters,
#                                                                 solar_clusters_ts,
#                                                                     resource_max_capacity=10)

# resource_clusters_wind,cluster_timeseries_wind=RES_module.select_top_sites(wind_clusters,
#                                                                 wind_clusters_ts,
#                                                                     resource_max_capacity=30)

In [ ]:
# import matplotlib.pyplot as plt
# import geopandas as gpd
# import pandas as pd

# legend_x_ax_offset=1

# # Ensure 'Region' is in the columns for both boundary and cells
# if 'Region' not in boundary.columns:
#     boundary = boundary.reset_index(inplace=True)

# # Assign a number to each region
# boundary['Region_Number'] = range(1, len(boundary) + 1)

# # Define custom bins and labels for solar and wind capacity
# solar_bins = [0, 100, 200, 300, 500, float('inf')]  # Custom ranges
# solar_labels = ['<100','100-200', '200-300', '300-500','>500']  # Labels for legend

# # Define custom bins and labels for solar and wind capacity
# wind_bins = [0, 300, 500, 1000, 2000,3000, float('inf')]  # Custom ranges
# wind_labels = ['<300','300-500', '500-1000', '1000-2000','2000-3000', '>3000']  # Labels for legend

# # Categorize potential_capacity_solar and potential_capacity_wind into bins
# resource_clusters_solar['solar_category'] = pd.cut(resource_clusters_solar['potential_capacity'], bins=solar_bins, labels=solar_labels, include_lowest=True)
# resource_clusters_wind['wind_category'] = pd.cut(resource_clusters_wind['potential_capacity'], bins=wind_bins, labels=wind_labels, include_lowest=True)

# # Create figure and axes for side-by-side plotting
# fig, (ax1, ax2) = plt.subplots(figsize=(18, 8), ncols=2)
# fig.suptitle("Potential Sites for Targeted Capacity Investments", fontsize=16,weight='bold')
# # Set axis off for both subplots
# ax1.set_axis_off()
# ax2.set_axis_off()

# # Shadow effect offset
# shadow_offset = 0.01

# # Plot solar map on ax1
# # Add shadow effect for solar map
# boundary.geometry = boundary.geometry.translate(xoff=shadow_offset, yoff=-shadow_offset)
# boundary.plot(ax=ax1, color='None', edgecolor='grey', linewidth=1, alpha=0.7)  # Shadow layer
# boundary.geometry = boundary.geometry.translate(xoff=-shadow_offset, yoff=shadow_offset)

# # Plot solar cells
# resource_clusters_solar.plot(column='solar_category', ax=ax1, cmap='Wistia', legend=True, 
#            legend_kwds={'title': "Solar Capacity (MW)", 'loc': 'upper right','fontsize':12,'bbox_to_anchor':(legend_x_ax_offset,1), 'frameon': False})

# # Plot actual boundary for solar map
# boundary.plot(ax=ax1, facecolor='none', edgecolor='black', linewidth=0.2, alpha=0.7)
# """
# # Annotate region numbers for solar map
# for idx, row in boundary.iterrows():
#     centroid = row.geometry.centroid
#     ax1.annotate(f"{row['Region_Number']}", 
#                  xy=(centroid.x, centroid.y), 
#                  ha='center', va='center',
#                  fontsize=7, color='black',
#                  bbox=dict(facecolor='white', edgecolor='none', alpha=0.7, boxstyle='round,pad=0.2'))
# """
# # Plot wind map on ax2
# # Add shadow effect for wind map
# boundary.geometry = boundary.geometry.translate(xoff=shadow_offset, yoff=-shadow_offset)
# boundary.plot(ax=ax2, color='None', edgecolor='grey', linewidth=1, alpha=0.7)  # Shadow layer
# boundary.geometry = boundary.geometry.translate(xoff=-shadow_offset, yoff=shadow_offset)

# # Plot wind cells
# resource_clusters_wind.plot(column='wind_category', ax=ax2, cmap='summer', legend=True, 
#            legend_kwds={'title': "Wind Capacity (MW)", 'fontsize':12,'bbox_to_anchor':(legend_x_ax_offset,1), 'frameon': False})

# # Plot actual boundary for wind map
# boundary.plot(ax=ax2, facecolor='none', edgecolor='black', linewidth=0.2, alpha=0.7)
# """
# # Annotate region numbers for wind map
# for idx, row in boundary.iterrows():
#     centroid = row.geometry.centroid
#     ax2.annotate(f"{row['Region_Number']}", 
#                  xy=(centroid.x, centroid.y), 
#                  ha='center', va='center',
#                  fontsize=8, color='black',
#                  bbox=dict(facecolor='white', edgecolor='none', alpha=0.7, boxstyle='round,pad=0.2'))
# """
# # Adjust layout for cleaner appearance
# fig.patch.set_alpha(0)  # Make figure background transparent
# plt.tight_layout()


# # Add annotation for solar capacity
# ax1.annotate(f"Targeted Capacity: \n{int(resource_clusters_solar.potential_capacity.sum()/1e3)} GW",
#              xy=(0.9, 0.6), xycoords='axes fraction', ha='center', 
#              fontsize=14, color='black', fontweight='bold')

# # Add annotation for wind capacity
# ax2.annotate(f"Targeted Capacity: \n{int(resource_clusters_wind.potential_capacity.sum()/1e3)} GW",
#              xy=(0.9, 0.6), xycoords='axes fraction', ha='center', 
#              fontsize=14, color='black', fontweight='bold')
# # Show the side-by-side plot

# plt.savefig('solar_wind_capacity_map.png',dpi=300)
# plt.show()

# Energy Calculations

### Proximity Range Filters

In [ ]:
# GRID_PROXIMITY_KM:int=30 #km
# cells_grid_proximity_filtered=cells[cells['nearest_station_distance_km']<=GRID_PROXIMITY_KM]

# solar_total_filtered=cells_grid_proximity_filtered['potential_capacity_solar'].sum()/1E3
# wind_total_filtered=cells_grid_proximity_filtered['potential_capacity_wind'].sum()/1E3

# print(f"Total potential solar capacity @ {GRID_PROXIMITY_KM}km grid proximity: {solar_total_filtered:.2f} GW")
# print(f"Total potential wind capacity @ {GRID_PROXIMITY_KM}km grid proximity: {wind_total_filtered:.2f} GW")

## Supply Curve Analytics

- Analysing 10000 GWh solar resource's supply curve

In [ ]:
for policy in POLICYs:
    
    cells=BC_dfs_all_runs[f'{policy}']['cells']
    
    print(f"Policy: {policy}")
    print("--" * 50)
    cells['potential_energy_solar_GWh']=cells['potential_capacity_solar']*cells['solar_CF_mean']*8760/1E3 # GWh
    cells['potential_energy_wind_GWh']=cells['potential_capacity_wind']*cells['wind_CF_mean']*8760/1E3 # GWh

    wind_total=cells['potential_capacity_wind'].sum()/1E3
    solar_total=cells['potential_capacity_solar'].sum()/1E3

    print(f"Total potential solar capacity: {solar_total:.2f} GW")
    print(f"Total potential wind capacity: {wind_total:.2f} GW\n")
    print("." * 10)
    
    cells_solar_sorted = cells.sort_values(by='lcoe_solar')
    solar_cumsum = cells_solar_sorted['potential_energy_solar_GWh'].cumsum()
    print("<=10,000 GWh from Solar")
    print("." * 10)
    cells_until_10000_solar = cells_solar_sorted.loc[solar_cumsum <= 10000]

    print(f"Total potential solar energy: {cells_until_10000_solar['potential_energy_solar_GWh'].sum():.2f} GWh")
    print(f"Number of cells needed (solar): {len(cells_until_10000_solar)}")
    print(f"{cells_until_10000_solar.potential_capacity_solar.sum()/1E3:.2f} GW potential solar capacity")
    print(f"Mean Score : {cells_until_10000_solar.lcoe_solar.mean():.2f} $/MWH")
    cells_until_10000_solar.lcoe_solar.describe()
    print("." * 10)
    cells_wind_sorted = cells.sort_values(by='lcoe_wind')
    wind_cumsum = cells_wind_sorted['potential_energy_wind_GWh'].cumsum()
    cells_until_40000_wind = cells_wind_sorted.loc[wind_cumsum <= 40000]
    print("<=40,000 GWh from Wind")
    print("." * 10)
    print(f"Total potential wind energy: {cells_until_40000_wind['potential_energy_wind_GWh'].sum():.2f} GWh")
    
    print(f"Number of cells needed (wind): {len(cells_until_40000_wind)}")
    print(f" {cells_until_40000_wind.potential_capacity_wind.sum()/1E3:.2f} GW potential wind capacity")
    print(f" Mean Score : {cells_until_40000_wind.lcoe_wind.mean():.2f} $/MWH")
    cells_until_40000_wind.lcoe_wind.describe()
    print('\n')
    
    
# cells_solar_sorted = cells.sort_values(by='lcoe_solar')
# solar_cumsum = cells_solar_sorted['potential_energy_solar_GWh'].cumsum()
# cells_until_10000_solar = cells_solar_sorted.loc[solar_cumsum <= 10000]

# print(f"Total potential solar energy: {cells_until_10000_solar['potential_energy_solar_GWh'].sum():.2f} GWh")
# print(f"Number of cells needed (solar): {len(cells_until_10000_solar)}")
# print(f" {cells_until_10000_solar.potential_capacity_solar.sum()/1E3:.2f} GW potential solar capacity")
# print(f" Mean Score : {cells_until_10000_solar.lcoe_solar.mean():.2f} $/MWH")
# cells_until_10000_solar.lcoe_solar.describe()

- Analysing 40000 GWh wind resource's supply curve

In [ ]:
lcoe_threshold_solar:int=57 #$/MWH
lcoe_threshold_wind:int=50 #$/MWH
for policy in POLICYs:
    print(f"Policy: {policy}")
    print("--" * 50)
    cells=BC_dfs_all_runs[f'{policy}']['cells']
    
    cells['potential_energy_solar_GWh']=cells['potential_capacity_solar']*cells['solar_CF_mean']*8760/1E3 # GWh
    cells['potential_energy_wind_GWh']=cells['potential_capacity_wind']*cells['wind_CF_mean']*8760/1E3 # GWh

    wind_total=cells['potential_capacity_wind'].sum()/1E3
    solar_total=cells['potential_capacity_solar'].sum()/1E3

    print(f"Total potential solar capacity: {solar_total:.2f} GW")
    print(f"Total potential wind capacity: {wind_total:.2f} GW\n")
    print("." * 10)
    
    solar_cells_filtered = cells[cells['lcoe_solar'] <= lcoe_threshold_solar]
    wind_cells_filtered = cells[cells['lcoe_wind'] <= lcoe_threshold_wind]
    
    print(f"Total potential solar capacity <={lcoe_threshold_solar} $/MWh LCOE threshold: {solar_cells_filtered['potential_capacity_solar'].sum()/1E3:.2f} GW")
    print(f"Total potential wind capacity <={lcoe_threshold_wind} $/MWh LCOE threshold: { wind_cells_filtered['potential_capacity_wind'].sum()/1E3:.2f} GW")


    print(f"Total potential solar energy @ <={lcoe_threshold_solar} $/MWh LCOE threshold: {solar_cells_filtered['potential_energy_solar_GWh'].sum():.2f} GWh")
    print(f"Total potential wind energy @ <={lcoe_threshold_wind} $/MWh LCOE threshold: {wind_cells_filtered['potential_energy_wind_GWh'].sum():.2f} GWh")
    print('\n')
    
    if policy=='BASELINE':
        solar_cells_filtered.to_csv(f"{BASELINE_results_save_to}/solar_cells_below_{lcoe_threshold_solar}_$pMWh_{region_code}_{policy}.csv",index=False)
        wind_cells_filtered.to_csv(f"{BASELINE_results_save_to}/wind_cells_below_{lcoe_threshold_wind}_$pMWh_{region_code}_{policy}.csv",index=False)
    else:
        solar_cells_filtered.to_csv(f"{POLICY_results_save_to}/solar_cells_below_{lcoe_threshold_solar}_$pMWh_{region_code}_{POLICY}.csv",index=False)
        wind_cells_filtered.to_csv(f"{POLICY_results_save_to}/wind_cells_below_{lcoe_threshold_wind}_$pMWh_{region_code}_{POLICY}.csv",index=False)


- Load BASELINE cells

In [ ]:
solar_cells_filtered_default= pd.read_csv(f"{BASELINE_results_save_to}/solar_cells_below_{lcoe_threshold_solar}_$pMWh_BC_BASELINE.csv")
wind_cells_filtered_default = pd.read_csv(f"{BASELINE_results_save_to}/wind_cells_below_{lcoe_threshold_wind}_$pMWh_BC_BASELINE.csv")

- Compare with Default (baseline)

In [ ]:
import matplotlib.pyplot as plt

# --- Prepare supply curve data (technical potential) ---
solar_sorted = solar_cells_filtered.sort_values(by='lcoe_solar')
solar_cumsum_energy = solar_sorted['potential_energy_solar_GWh'].cumsum()
solar_lcoe = solar_sorted['lcoe_solar']

wind_sorted = wind_cells_filtered.sort_values(by='lcoe_wind')
wind_cumsum_energy = wind_sorted['potential_energy_wind_GWh'].cumsum()
wind_lcoe = wind_sorted['lcoe_wind']

# --- Prepare policy-constrained curves ---
solar_sorted_default = solar_cells_filtered_default.sort_values(by='lcoe_solar')
solar_cumsum_energy_default = solar_sorted_default['potential_energy_solar_GWh'].cumsum()
solar_lcoe_default = solar_sorted_default['lcoe_solar']

wind_sorted_default = wind_cells_filtered_default.sort_values(by='lcoe_wind')
wind_cumsum_energy_default = wind_sorted_default['potential_energy_wind_GWh'].cumsum()
wind_lcoe_default = wind_sorted_default['lcoe_wind']

# --- Plot ---

fig, ax = plt.subplots(figsize=(7, 4), dpi=1000)

# Policy-constrained (emphasized solid lines)
ax.plot(
    solar_cumsum_energy, solar_lcoe,
    label='Solar (Policy-constrained)',
    color="#DA391D", linestyle='-', linewidth=2.2,alpha=0.8
)
ax.plot(
    wind_cumsum_energy, wind_lcoe,
    label='Wind (Policy-constrained)',
    color="#530c9a", linestyle='-', linewidth=2.2,alpha=0.8
)

# Baseline (lighter dashed lines)
ax.plot(
    solar_cumsum_energy_default, solar_lcoe_default,
    label='Solar (Baseline)',
    color="#F494346D", linestyle='--', linewidth=3.0, alpha=1,
    zorder=2
)
ax.plot(
    wind_cumsum_energy_default, wind_lcoe_default,
    label='Wind (Baseline)',
    color="#b165fc3c", linestyle='--', linewidth=3.0, alpha=1,
    zorder=2
)

# Labels and title
ax.set_xlabel('Cumulative Potential Energy (GWh)', fontsize=12, labelpad=8)
ax.set_ylabel('Relative Cost Score ($/MWh)', fontsize=9, labelpad=8)
ax.set_title(
    'Supply Curves for Solar and Wind Resources\nBaseline vs. Policy-Constrained Potential',
    fontsize=14, weight='bold', pad=12
)

# Optional axis limits (if you want to emphasize a range)
# ax.set_xlim(0, 20000)

# Legend
ax.legend(
    frameon=False,
    fontsize=12,
    ncol=2,
    loc='upper right',
    bbox_to_anchor=(1, 0.3),  # slightly lower for better spacing
    # title="Scenario",
    title_fontsize=11
)

# Clean spines
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)

# Gridlines for readability
ax.grid(True, which='both', linestyle=':', linewidth=0.7, color='gray', alpha=0.4)
# Set x-axis limit to 40,000 GWh
ax.set_xlim(0, 40000)
# Make tick mark font bigger
ax.tick_params(axis='both', which='major', labelsize=11)
# Tight layout
plt.tight_layout()
# plt.show()
plt.savefig(f"{common_vis_save_to}/supply_curve_baseline_vs_policy_{region_code}.svg", bbox_inches='tight', transparent=False)

plt.savefig(f"../docs/source/_static/supply_curve_baseline_vs_policy_{region_code}.png", bbox_inches="tight",transparent=True)

- Extract the co-located Resources (cells)

In [ ]:
common_idx = solar_cells_filtered.index.intersection(wind_cells_filtered.index)
solar_common_capacity = solar_cells_filtered.loc[common_idx, 'potential_capacity_solar'].sum() / 1E3
wind_common_capacity = wind_cells_filtered.loc[common_idx, 'potential_capacity_wind'].sum() / 1E3
timeseries_solar_filtered=BC_dfs_all_runs[f'{POLICY}']['timeseries_solar'][common_idx]
timeseries_wind_filtered=BC_dfs_all_runs[f'{POLICY}']['timeseries_wind'][common_idx]
print(f"Common index count: {len(common_idx)}")
print(f"Solar capacity (GW) in common cells: {solar_common_capacity:.2f}")
print(f"Wind capacity (GW) in common cells: {wind_common_capacity:.2f}")

#  Clusters

In [ ]:
vis.plot_region_complementarity(timeseries_solar=timeseries_clusters_solar, 
                                timeseries_wind=timeseries_clusters_wind, 
                                region_code=region_code,
                                # region='PeaceRiver',
                                RUN_ID=POLICY,
                                # aggregate=True,
                                show=True)


In [ ]:
# # Calculate standard deviation of difference for each cluster vs PeaceRiver_1
# std_devs = timeseries_clusters_wind.subtract(timeseries_clusters_wind['PeaceRiver_1'], axis=0).std()

# # Find the cluster with the maximum deviation
# most_deviated_cluster = std_devs.idxmax()
# max_deviation = std_devs.max()

# print(f"Most deviated cluster: {most_deviated_cluster} (std={max_deviation:.4f})")


In [ ]:
# timeseries_clusters_wind[most_deviated_cluster].to_csv(f"../results/temp/timeseries_clusters_wind_{RUN_ID}_{most_deviated_cluster}.csv")

In [ ]:
# # Compute daily means for PeaceRiver_1 and most_deviated_cluster
# peace_daily = timeseries_clusters_wind['PeaceRiver_1'].resample('1D').mean()
# deviated_daily = timeseries_clusters_wind[most_deviated_cluster].resample('1D').mean()

# fig, ax = plt.subplots(figsize=(10, 4), dpi=300)
# ax.plot(peace_daily.index, peace_daily.values, label='PeaceRiver_1', color='navy', linewidth=2)
# ax.plot(deviated_daily.index, deviated_daily.values, label=most_deviated_cluster, color='orangered', linewidth=2)
# ax.set_title(f"Daily Mean Wind CF: PeaceRiver_1 vs {most_deviated_cluster}", fontsize=14)
# ax.set_ylabel("Capacity Factor")
# ax.set_xlabel("Date")
# ax.legend(frameon=False)
# ax.grid(alpha=0.3, linestyle='--')
# plt.tight_layout()
# # plt.show()

--- 
# Background Illustrations

---

## Regional Outlines (not sensitive to Policies)

In [ ]:
import matplotlib
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(7, 5), dpi=500)
region_names_anchors = (-0.01, 0.88)  # adjust legend placement

unique_regions = boundary_plot["Region_Number"].unique()
N = len(unique_regions)

# Option A: plt.get_cmap with N discrete colors
base_cmap = plt.get_cmap("tab20", N)
colors = [base_cmap(i) for i in range(N)]

# shuffle for randomness
np.random.shuffle(colors)

region_color_map = dict(zip(unique_regions, colors))
boundary_plot["color"] = boundary_plot["Region_Number"].map(region_color_map)

# --- Plot polygons with color fill ---
boundary_plot.plot(ax=ax,
                   linewidth=0.4,
                   alpha=0.6,
                   facecolor=boundary_plot["color"],
                   edgecolor="white",
                   )

ax.set_axis_off()

# --- Annotate region numbers with white halo ---
for idx, row in boundary_plot.iterrows():
    if row.geometry is not None and not row.geometry.is_empty:
        x, y = row.geometry.centroid.x, row.geometry.centroid.y
        ax.text(
            x, y, str(row["Region_Number"]),
            ha="center", va="center",
            fontsize=10, fontweight="bold", color="black",
            path_effects=[pe.withStroke(linewidth=2, foreground="white",alpha=0.4)]
        )

# --- Legend ---
handles = [
    mpatches.Patch(facecolor=region_color_map[num], edgecolor="None",
                   label=f"{num} - {name}")
    for num, name in zip(region_mapping["Region_Number"], region_mapping["Region"])
]
ax.legend(handles=handles,
          bbox_to_anchor=region_names_anchors,
          loc="upper left", frameon=False, fontsize=6.5)

ax.set_facecolor("none")  # transparent background
plt.tight_layout()
plt.savefig(f"../vis/{country_kwd}/{region_code}/{region_code}_regions.png",
            bbox_inches="tight", transparent=True)


In [ ]:
if cells.crs!=boundary_plot.crs:
    cells = cells.to_crs(boundary_plot.crs)
fig, ax = plt.subplots(figsize=(7, 5),dpi=500, facecolor='none')
cells.boundary.plot(ax=ax, linewidth=0.5, color='k', alpha=1)
vis.add_compass_arrow_custom(ax,x=0.8,text_offset=0.03)
ax.set_axis_off()
# fig.patch.set_alpha(1)  # Make figure background transparent
# ax.set_facecolor('none')  # Make axis background transparent
plt.tight_layout()
plt.savefig(f"../vis/{country_kwd}/{region_code}/{region_code}_gridcells_outline.svg")
# plt.show()

## Grid

In [ ]:
lines=BC_dfs_all_runs[f'{POLICY}']['lines']
if lines.crs!=boundary_plot.crs:
    lines = lines.to_crs(boundary_plot.crs)

lines_cleaned=lines[lines['power']=='line']

In [ ]:
# for columns in lines.columns:
#     print(f"{columns}")

### lines

In [ ]:
def plot_grid_lines(
    region_code: str,
    region_name: str,
    lines: gpd.GeoDataFrame,
    boundary: gpd.GeoDataFrame,
    font_family: str = None,
    figsize: tuple = (10, 8),
    dpi=1000,
    save_to: str | Path = None,
    show: bool = True,
):
    """
    Plots transmission lines with binned voltage levels in a specified region.
    """
    lines = lines.copy() # avoid modifying original
    fig, ax = plt.subplots(figsize=figsize, dpi=dpi)
    fig.suptitle("Transmission Lines by Voltage Levels", fontsize=16, fontweight='bold')
    # plt.style.use(style_path)
    if font_family is not None:
        plt.rcParams['font.family'] = font_family

    boundary.plot(ax=ax, facecolor='White', edgecolor='black', linewidth=0.2, alpha=0.7)

    if 'voltage' in lines.columns:
        # Convert to numeric
        lines['voltage_kv'] = pd.to_numeric(lines['voltage'], errors='coerce') / 1000

        # Define voltage bins
        bins = [0, 12, 25, 132, 220, float("inf")]
        labels = ["<12 kV", "12–25 kV", "25–132 kV", "132–220 kV", "≥220 kV"]
        lines['voltage_class'] = pd.cut(lines['voltage_kv'], bins=bins, labels=labels, right=False)

        # Color map (enough distinct colors)
        cmap = plt.colormaps.get_cmap('tab10')
        colors = [cmap(i) for i in range(len(labels))]
        color_map = {label: colors[i] for i, label in enumerate(labels)}


        # Plot by class
        for label in labels:
            mask = lines['voltage_class'] == label
            if mask.any():
                lines[mask].plot(ax=ax, color=color_map[label], linewidth=1, alpha=0.8)

        # Legend
        legend_patches = [mpatches.Patch(color=color_map[label], label=label) for label in labels if label in lines['voltage_class'].unique()]
        # ax.legend(handles=legend_patches, frameon=False, fontsize=11, loc='upper right')

    else:
        lines.plot(ax=ax, color='blue', linewidth=1,alpha=0.7)

        
    # =============================
    # 4️⃣ Add existing solar/wind projects
    # =============================
    existing_VREs_gdf_solar = existing_VREs_plot[existing_VREs_plot['Technology'].str.lower() == 'solar']
    ax1, solar_legends = vis.get_existing_committed_VRE_plot(
        ax=ax,
        existing_VREs_gdf=existing_VREs_gdf_solar,
        existing_VRE_type_column='Technology',
        committed_VREs_gdf=committed_VREs_plot,
        committed_VRE_type_column='Technology',
        target_crs=CRS_m,
        marker_scale_existing=0.2,
        marker_scale_committed=0.2,
        marker_highlight_width=4
    )

    existing_VREs_gdf_wind = existing_VREs_plot[existing_VREs_plot['Technology'].str.lower() == 'wind']
    ax2, wind_legends = vis.get_existing_committed_VRE_plot(
        ax=ax,
        existing_VREs_gdf=existing_VREs_gdf_wind,
        existing_VRE_type_column='Technology',
        committed_VREs_gdf=committed_VREs_plot,
        committed_VRE_type_column='Technology',
        target_crs=CRS_m,
        marker_scale_existing=0.2,
        marker_scale_committed=0.2,
        marker_highlight_width=4
    )
    # =============================
    # Add legends and note
    # =============================
    # Combine legend handles from both plots
    combined_legend_handles = legend_patches + solar_legends + wind_legends

    # Extract labels
    labels = [h.get_label() for h in combined_legend_handles]

    # --- Deduplicate using dictionary (preserves order) ---
    unique = dict(zip(labels, combined_legend_handles))
    unique_labels = list(unique.keys())
    unique_handles = list(unique.values())
    wrapped_labels = [textwrap.fill(label, width=30) for label in unique_labels]

    fig.legend(
        handles=unique_handles,
        labels=wrapped_labels,
        loc='upper right',
        bbox_to_anchor=(0.8, 0.85),
        ncol=1,
        fontsize=8,
        frameon=False,
        handlelength=1.5,    # horizontal length of the legend handle
        handleheight=1.2,    # vertical spacing
        markerscale=0.7      # scales marker size relative to the plot markers
    )
    
    
    ax.set_axis_off()
    plt.tight_layout()

    if save_to is None:
        save_to = Path("vis") / region_code / "network"
    else:
        save_to = Path(save_to)
    
    save_to.mkdir(parents=True, exist_ok=True)
    save_to_file = save_to / f"transmission_lines_{region_code}.svg"
    plt.savefig(save_to_file, bbox_inches='tight', dpi=300,transparent=True)
    
    utils.print_update(level=2, message=f"Transmission Lines for {region_name} saved to {save_to_file}")
    if show:
        plt.show()
    return lines

In [ ]:
lines_new=plot_grid_lines(
    region_code=region_code,
    region_name=region_name,
    lines=lines_cleaned,
    boundary=boundary_plot,
    figsize=(7, 5),
    dpi=1000,
    save_to=common_vis_save_to,
    show=True
)

### Grid Proximity

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

# Define custom bins and labels for solar and wind capacity
bins = [0, 5, 20, 50, 100, float('inf')]  # Custom ranges
labels = ['<5','5-20', '20-50', '50-100', '>100']  # Labels for legend

if cells_scenario.crs!=boundary_plot.crs:
    cells_proj= cells.to_crs(boundary_plot.crs)

# Categorize potential_capacity_solar and potential_capacity_wind into bins
cells_proj['station_distance_category'] = pd.cut(cells['nearest_station_distance_km'], bins=bins, labels=labels, include_lowest=True)


# Create figure and axes for side-by-side plotting
fig, (ax) = plt.subplots(figsize=(7, 5),dpi=1000)
fig.suptitle("Proximity to Existing Grid Nodes", fontsize=16, fontweight='bold',y=0.95)
ax.set_axis_off()

# Shadow effect offset
shadow_offset = 0.002

# Plot solar map on ax1
# Add shadow effect for solar map
# boundary_plot.geometry = boundary_plot.geometry.translate(xoff=shadow_offset, yoff=-shadow_offset)
boundary_plot.plot(ax=ax, facecolor='none', edgecolor='gray', linewidth=0.2, alpha=0.4)  # Shadow layer
# boundary_plot.geometry = boundary_plot.geometry.translate(xoff=-shadow_offset, yoff=shadow_offset)

# Plot solar cells
cells_proj.plot(
    column='station_distance_category',
    ax=ax,
    cmap='copper',
    legend=True,
    linewidth=0.1,
    legend_kwds={
        'title': "Nearest sub-stations (km)",
        'title_fontsize': 11,
        'loc': 'upper right',
        'bbox_to_anchor': (0.38, 0.35),
        'fontsize':10.5,
        'frameon': False
    }
)

# Plot actual boundary for solar map
boundary_plot.plot(ax=ax, facecolor='None', edgecolor='black', linewidth=0.5, alpha=0.9)
# =============================
# 4️⃣ Add existing solar/wind projects
# =============================
existing_VREs_gdf_solar = existing_VREs_plot[existing_VREs_plot['Technology'].str.lower() == 'solar']
ax1, solar_legends = vis.get_existing_committed_VRE_plot(
    ax=ax,
    existing_VREs_gdf=existing_VREs_gdf_solar,
    existing_VRE_type_column='Technology',
    committed_VREs_gdf=committed_VREs_plot,
    committed_VRE_type_column='Technology',
    target_crs=CRS_m,
    marker_scale_existing=0.2,
    marker_scale_committed=0.2,
    marker_highlight_width=4
)

existing_VREs_gdf_wind = existing_VREs_plot[existing_VREs_plot['Technology'].str.lower() == 'wind']
ax2, wind_legends = vis.get_existing_committed_VRE_plot(
    ax=ax,
    existing_VREs_gdf=existing_VREs_gdf_wind,
    existing_VRE_type_column='Technology',
    committed_VREs_gdf=committed_VREs_plot,
    committed_VRE_type_column='Technology',
    target_crs=CRS_m,
    marker_scale_existing=0.2,
    marker_scale_committed=0.2,
    marker_highlight_width=4
)
# =============================
# Add legends and note
# =============================
# Combine legend handles from both plots
# combined_legend_handles = solar_legends + wind_legends

# # Extract labels
# labels = [h.get_label() for h in combined_legend_handles]

# # --- Deduplicate using dictionary (preserves order) ---
# unique = dict(zip(labels, combined_legend_handles))
# unique_labels = list(unique.keys())
# unique_handles = list(unique.values())
# wrapped_labels = [textwrap.fill(label, width=30) for label in unique_labels]

# fig.legend(
#     handles=unique_handles,
#     labels=wrapped_labels,
#     loc='upper right',
#     bbox_to_anchor=(0.8, 0.85),
#     ncol=1,
#     fontsize=8,
#     frameon=False,
#     handlelength=1.5,    # horizontal length of the legend handle
#     handleheight=1.2,    # vertical spacing
#     markerscale=0.7      # scales marker size relative to the plot markers
# )

# Adjust layout for cleaner appearance
# fig.patch.set_alpha(0)  # Make figure background transparent
plt.tight_layout()
save_to = Path(common_vis_save_to/f"Resources_proximity_to_grid_{region_code}.png",transparent=True)
plt.savefig(save_to, bbox_inches='tight')
# plt.savefig(f"../docs/source/_static/Resources_proximity_to_grid_{region_code}.jpg", bbox_inches='tight')

# GWA raster vs ERA5

* Plot wind CF raster (benchmark) vs Calculated CF for ERA5 Cells

In [ ]:
gwa_country_code=cfg_policy.get('region_mapping').get(region_code).get('GWA_country_code')
utils.print_banner(f"GWA Country Code Selected: {gwa_country_code}")

In [ ]:
import rioxarray as rxr

raster_path=f'../data/downloaded_data/GWA/{gwa_country_code}_capacity-factor_IEC3.tif'
gwa_raster_data = (
        rxr.open_rasterio(raster_path)
        .rio.clip_box(**bounding_box_dict)
        .rename('CF_IEC3')
        .isel(band=1 if '*Class*' in 'CF_IEC3' else 0)  # 'IEC_Class_ExLoads' data is in band 1
        # .drop_vars(['band', 'spatial_ref']) # removes CRS info
    )

In [ ]:
# import matplotlib.pyplot as plt

# fig, ax = plt.subplots(figsize=(3.5, 2.5),dpi=500)
# gwa_raster_data.plot(ax=ax, cmap='BuPu', add_colorbar=True)
# boundary.plot(ax=ax, facecolor='none', edgecolor='white', linewidth=0.5)
# ax.set_title("GWA CF-IEC3 Reference (High-res)")
# ax.axis('off')
# plt.savefig(f"../vis/{region_code}/GWA_CF_IEC3.png", bbox_inches='tight', transparent=False)

In [ ]:
if gwa_raster_data.rio.crs != CRS_m:
    gwa_raster_plot = gwa_raster_data.rio.reproject(CRS_m)   # <- fixed typo
else:
    gwa_raster_plot = gwa_raster_data
    
if cells_scenario.crs != CRS_m:
    cells_scenario_proj=cells_scenario.to_crs(CRS_m)
else:
    cells_scenario_proj=cells_scenario

In [ ]:
# vis.get_CF_wind_check_plot(cells_baseline, 
#                        gwa_raster_plot,
#                        boundary_plot,
#                        region_code,
#                        region_name,
#                        ['CF_IEC3', 'wind_CF_mean'],
#                        font_family='sans-serif',
#                        figure_height=4,
#                        save_to=f"{BASELINE_vis_save_to}/GWA_CF_IEC3_vs_cells_wind_CF_mean_{region_code}.png")

In [ ]:
vis.get_CF_wind_check_plot(cells_scenario_proj, 
                       gwa_raster_plot,
                       boundary_plot,
                       region_code,
                       region_name,
                       ['CF_IEC3', 'wind_CF_mean'],
                       font_family='sans-serif',
                       figure_height=5,
                       save_to=common_vis_save_to)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set a clean and minimal style
sns.set_style("white")

# Initialize the figure
plt.figure(figsize=(7.5, 2.5),dpi=1000)
plt.style.use('../RES/visual_styles/elsevier.mplstyle')

# Create the boxplot
ax = sns.boxplot(
    data=cells_scenario[['CF_IEC2', 'CF_IEC3', 'wind_CF_mean']],
    palette="Paired",
    linewidth=0.2,
    width=0.6
)

# Set title and labels
ax.set_title('Capacity Factor Distribution Comparison', weight='semibold', pad=12)
ax.set_ylabel('Capacity Factor')
ax.set_xlabel('')

# Tweak tick formatting
ax.tick_params(axis='x')
ax.tick_params(axis='y')

# Remove all spines
for spine in ax.spines.values():
    spine.set_visible(False)

# Add horizontal grid lines
ax.yaxis.grid(True, linestyle='--', alpha=0.4)
ax.xaxis.grid(False)

plt.figtext(
    0.01, -0.08,
    "*'CF_IEC2, CF_IEC3: Average CF for ERA5 Cells, calculated from high-resolution yearly average CF from GWA for IEC Class 2 and 3 turbines.\n"
    "*wind_CF_mean: Average CF for ERA5 Cells, calculated from the ERA5 windspeed (rescaled with GWA) time series ",
    ha='left', fontsize=7, style='normal', fontweight='normal', color='gray',wrap=True
)

plt.tight_layout()
plt.savefig('../docs/source/_static/CF_distribution_comparison.png', bbox_inches="tight")
plt.savefig(f"{common_vis_save_to}/CF_distribution_comparison.png", bbox_inches='tight', transparent=False)

# Lands
---

## Rasters

### Canadian Gov
> Loaded as Custom Raster 

In [ ]:
custom_rasters:list=cfg_policy.get('custom_land_layers').get('rasters')

#### Landcover

- [Source1](https://app.geo.ca/en-ca/map-browser/record/ee1580ab-a23d-4f86-a09b-79763677eb47) or [Source2](https://open.canada.ca/data/en/dataset/ee1580ab-a23d-4f86-a09b-79763677eb47)
- [About data](https://osdp-psdo.canada.ca/dp/en/search/metadata/NRCAN-FGP-1-ee1580ab-a23d-4f86-a09b-79763677eb47)
- [About classes](https://developers.google.com/earth-engine/datasets/catalog/USGS_NLCD_RELEASES_2020_REL_NALCMS), and [more](https://publications.gc.ca/collections/collection_2025/statcan/16-510-x2025001-eng.pdf?utm_source=chatgpt.com)

- Load Raster (clipped to BC and 100m Resolution), legends

In [ ]:
CanGov_landcover_path='../data/downloaded_data/Gov/LandCover/landcover-2020-classification_clipped_BC_100m.tif'
CanGov_landcover_legends=pd.read_csv('../data/legends/LandCover_CANgov_2020_legend.csv')
CanGov_landcover_da=utils.get_raster_da(CanGov_landcover_path)

- Review Existing facilities and Landcover mapping

In [ ]:
existing_VREs_gdf_with_landcover,existing_VREs_summary = lands.assign_raster_class_to_points(
    gdf=existing_VREs_gdf,
    raster_da=CanGov_landcover_da,
    legend_df=CanGov_landcover_legends,
    class_col_name="CanGov_landcover",
    resource_type_col_name='gen_type'
)

vis.plot_vre_sites_by_landcover(
    df=existing_VREs_summary,
    class_col="CanGov_landcover_description",
    count_prefix="SiteCount_",
    title="Existing VRE Sites by Land-Cover Class and Technology",
    figsize=(6, 2),
    wrap_width=25,
    fontsize=7,
    # normalize=True,
    colors=["#2e0354", "#e0b732"],
    save_to=common_vis_save_to/"Existing_VREs_by_LandCover_CANgov2020.png"
)

- Review class distribution and find the dominant classes

In [ ]:
LandCover_CANgov2020_class_distribution=lands.plot_raster_class_distribution(CanGov_landcover_da,
                                     CanGov_landcover_legends,
                                     show=True,
                                     figsize=(8, 4),
                                    #  pct_threshold=0.05,
                                     save_path=common_vis_save_to/f'LandCover_CANgov_class_distribution_{region_code}.png')
LandCover_CANgov2020_class_distribution.sort_values(by='Percentage',ascending=False,inplace=True)
LandCover_CANgov2020_class_distribution

In [ ]:
selected_classes_solar=[8, 10, 13, 15, 16, 17]
LC_test=LandCover_CANgov2020_class_distribution[LandCover_CANgov2020_class_distribution['class'].isin(selected_classes_solar)]

In [ ]:
LC_test

In [ ]:
LandCover_CANgov2020_class_distribution

- Define layers for plotting in Map

In [ ]:
land_cover_cfg= next((item for item in cfg_policy.get('custom_land_layers').get('rasters') if 'CanGov_LandCover_2020' in item.get('name', '')), None)
layers_included:dict=land_cover_cfg['class_inclusion']

- Review the spatial filters

In [ ]:
classes_to_plot=layers_included # None, for all classes

if classes_to_plot is None: 
    utils.print_warning("All classes are being plotted. This may lead to a cluttered visualization.")
    save_to_plot = f"{common_vis_save_to}/CanGov_Landcover_AllClasses_with_existing_VREs.png"

else:
    save_to_plot = f"{POLICY_vis_save_to}/CanGov_Landcover_SelectedClasses_with_existing_VREs.png"
    utils.print_update(level=1, message=f"Plotting selected classes: {classes_to_plot}")

fig1,ax1,save_to1=vis.plot_developable_land_and_vres(
    target_crs=CRS_m,
    raster_data=CanGov_landcover_da,
    raster_legends=CanGov_landcover_legends,
    classes_to_plot=classes_to_plot, #layers,                 # <- all classes
    boundary=boundary,
    existing_VREs_gdf=existing_VREs_gdf,
    committed_VREs_gdf=committed_VREs_gdf,
    marker_scale_existing=5.0,
    marker_scale_committed=0.4,
    dpi=1000,
    marker_highlight_width=3,
    title="Suitable Landcovers (Canada Gov 2020)",
    existing_marker_col="wind_turbine_capacity",
    output_path=save_to_plot,
    legend_anchor=(.73, .98)
)

utils.print_update(level=1,message=f"plot saved to:: {save_to_plot}")

# # Update DOC contents (if needed)
# doc_save_to="../docs/source/_static/CanGov_Landcover_with_existing_VREs.png"
# fig1.savefig(doc_save_to, bbox_inches="tight")

### GAEZ

#### Landcover
> Replaced by CanGov Landcover in Canadian Study

In [ ]:
# GAEZ_landcover_raster_path=f'../data/downloaded_data/GAEZ/Rasters_in_use/LR/ter/slpmed30s_clipped_{region_code}.tif'
# GAEZ_landcover_raster_da = utils.get_raster_da(GAEZ_landcover_raster_path)

# GAEZ_landcover_raster_legends=pd.read_csv("../data/legends/gaez_landcover_legend.csv")

In [ ]:
# land_cover_cfg= next((item for item in cfg_policy.get('GAEZ').get('raster_types') if 'land_cover' in item.get('name', '')), None)
# if land_cover_cfg is None:
#     raise ValueError("[ERROR] 'land_cover' configuration not found in config file.")
# else:
#     layers_included:dict=land_cover_cfg['class_inclusion']

#     lands.plot_raster_class_distribution(GAEZ_landcover_raster_da,
#                                         GAEZ_landcover_raster_legends,
#                                         show=True,
#                                         save_path=f'../vis/{country_kwd}/{region_code}/GAEZ_landcover_class_distribution_{region_code}.png')

#     # Unique CLC codes in your clipped area
#     unique_classes = np.unique(GAEZ_landcover_raster_da.values[~np.isnan(GAEZ_landcover_raster_da.values)])

#     layers_excluded = {
#         tech: [c for c in unique_classes if c not in layers_included[tech] and c != 0]
#         for tech in ['solar', 'wind']
#     }

#     classes_to_plot = layers_included

#     fig,ax,save_to=vis.plot_developable_land_and_vres(
#         target_crs=CRS_m,
#         raster_data=GAEZ_landcover_raster_da,
#         raster_legends=GAEZ_landcover_raster_legends,
#         classes_to_plot=classes_to_plot, #layers_included,                 # <- all classes
#         boundary=boundary,
#         existing_VREs_gdf=existing_VREs_gdf,
#         committed_VREs_gdf=committed_VREs_gdf,
#             marker_scale_existing=5.0,
#         marker_scale_committed=0.4,
#         title="Terrains (GAEZ_v4 slop median 0.5)",
#         existing_marker_col="wind_turbine_capacity",
#         output_path=f"../vis/{country_kwd}/{region_code}/GAEZ_terrains_with_existing_VREs.png",
#         legend_anchor=(.73, 1)
#     )

#     if classes_to_plot is None:
#         utils.print_warning("[WARN] All classes are being plotted. This may lead to a cluttered visualization.")
#         fig.savefig(f"{POLICY_vis_save_to}/GAEZ_landcover_AllClasses_with_existing_VREs.png", bbox_inches="tight")
#     else:
#         utils.print_update(level=1, message=f"[INFO] Plotting specified classes: {classes_to_plot}")
#         # Update DOC contents (if needed)
#         doc_save_to="../docs/source/_static/GAEZ_landcover_with_existing_VREs.png"
#         fig.savefig(doc_save_to, bbox_inches="tight")
#         utils.print_update(level=1,message=f"plot saved to: {doc_save_to}")
#         fig.savefig(f"{POLICY_vis_save_to}/GAEZ_landcover_SuitableLands_with_existing_VREs.png", bbox_inches="tight")
#         utils.print_update(level=1,message=f"plot saved to: {POLICY_vis_save_to}/GAEZ_landcover_SuitableLands_with_existing_VREs.png")

#### Terrain

- Define attributes from Config, legends

In [ ]:
GAEZ_terrain_raster_legends=pd.read_csv("../data/legends/gaez_terrains_legend.csv")
terrain_cfg= next((item for item in cfg_policy.get('GAEZ').get('raster_types') if 'terrain_resources' in item.get('name', '')), None)

- Load Raster (clipped to BC), 

In [ ]:
GAEZ_terrain_raster_path=f'../data/downloaded_data/GAEZ/Rasters_in_use/LR/ter/slpmed30s_clipped_{region_code}.tif'
GAEZ_terrain_raster_da = utils.get_raster_da(GAEZ_terrain_raster_path)

In [ ]:
#GAEZ_terrain_raster_path= lands.clip_to_boundary_and_resample_raster(
#         in_raster_config=terrain_cfg,
#         source_raster_path=f'../data/downloaded_data/GAEZ/Rasters_in_use/LR/ter/slpmed30s.tif',
#         boundary_name=region_code,
#         boundary=boundary,
#         CRS_meters=CRS_m
# )

- Review the existing sites mapping to terrain layers

In [ ]:
lands.plot_raster_class_distribution(GAEZ_terrain_raster_da,
                                     GAEZ_terrain_raster_legends,
                                     show=True,
                                     figsize=(9, 3),
                                     save_path=common_vis_save_to/f'GAEZ_terrains_class_distribution_{region_code}.png')


- Review Existing facilities' mapping to layers

In [ ]:
existing_VREs_gdf_with_landcover,existing_VREs_summary = lands.assign_raster_class_to_points(
    gdf=existing_VREs_gdf,
    raster_da=GAEZ_terrain_raster_da,
    legend_df=GAEZ_terrain_raster_legends,
    class_col_name="GAEZ_terrain",
    resource_type_col_name='gen_type'
)

vis.plot_vre_sites_by_landcover(
    df=existing_VREs_summary,
    class_col="GAEZ_terrain_description",
    count_prefix="SiteCount_",
    title="Existing VRE Sites by Terrain Classes and Technology",
    figsize=(8, 2),
    wrap_width=25,
    fontsize=7,
    # normalize=True,
    colors=["#2e0354", "#e0b732"],
    save_to=common_vis_save_to/"Existing_VREs_by_Terrains_GAEZ.png"
)

- Load layers to be plotted in map

In [ ]:
layers_excluded:dict=terrain_cfg['class_exclusion']

# Unique CLC codes in your clipped area
unique_classes = np.unique(GAEZ_terrain_raster_da.values[~np.isnan(GAEZ_terrain_raster_da.values)])

layers_included = {
    tech: [c for c in unique_classes if c not in layers_excluded[tech] and c != 0]
    for tech in ['solar', 'wind']
}

- Plot in Map

In [ ]:
classes_to_plot = layers_included #None

if classes_to_plot is None or all(len(v) == 0 for v in layers_included.values()):
    title="Terrains All Layers (GAEZ_v4 Terrain slop median 0.5)",
    save_to_plot = common_vis_save_to/'GAEZ_terrains_AllClasses_with_existing_VREs.png'
    utils.print_warning("[WARN] All classes are being plotted. This may lead to a cluttered visualization.")
else:
    title="Terrains Selected Classes (GAEZ_v4 slop median 0.5)",
    save_to_plot = f"{POLICY_vis_save_to}/GAEZ_terrains_SelectedClasses_with_existing_VREs.png"
    utils.print_update(level=1, message=f"Plotting selected classes: {layers_included}")

fig2,ax2,save_to2=vis.plot_developable_land_and_vres(
    target_crs=CRS_m,
    raster_data=GAEZ_terrain_raster_da,
    raster_legends=GAEZ_terrain_raster_legends,
    classes_to_plot=classes_to_plot, #layers_included,                 # <- all classes
    boundary=boundary,
    existing_VREs_gdf=existing_VREs_gdf,
    committed_VREs_gdf=committed_VREs_gdf,
        marker_scale_existing=5.0,
    marker_scale_committed=0.4,
    dpi=1000,
    title=title,
    existing_marker_col="wind_turbine_capacity",
    output_path=save_to_plot,
    legend_anchor=(.73, .8),
    marker_highlight_width=3,
)

utils.print_update(level=1,message=f"plot saved to:: {save_to_plot}")

#### Exclusions

- Load Legends, config, raster path

In [ ]:
excld_raster_legends=pd.read_csv("../data/legends/gaez_exclusion_legend.csv")
excld_cover_cfg= next((item for item in cfg_policy.get('GAEZ').get('raster_types') if 'exclusion_areas' in item.get('name', '')), None)

- Load Raster

In [ ]:
# excld_raster_path=lands.clip_to_boundary_and_resample_raster(
#         in_raster_config=terrain_cfg,
#          source_raster_path='../data/downloaded_data/GAEZ/Rasters_in_use/LR/excl/exclusion_2017.tif',
#          boundary_name=region_code,
#          boundary=boundary,
#          CRS_meters=CRS_m
# )

In [ ]:
excld_raster_path = f"../data/downloaded_data/GAEZ/Rasters_in_use/LR/excl/exclusion_2017_clipped_{region_code}.tif"
excld_raster_data = utils.get_raster_da(excld_raster_path)

- Review Class Distribution of raster

In [ ]:
lands.plot_raster_class_distribution(excld_raster_data,
                                     excld_raster_legends,
                                     show=True,
                                     figsize=(9, 2.5),
                                     save_path=common_vis_save_to/f'GAEZ_Exclusion_class_distribution_{region_code}.png')

- Review Existing sites mapping to layers

In [ ]:
existing_VREs_gdf_with_landcover,existing_VREs_summary = lands.assign_raster_class_to_points(
    gdf=existing_VREs_gdf,
    raster_da=excld_raster_data,
    legend_df=excld_raster_legends,
    class_col_name="GAEZ_exclusion",
    resource_type_col_name='gen_type'
)

vis.plot_vre_sites_by_landcover(
    df=existing_VREs_summary,
    class_col="GAEZ_exclusion_description",
    count_prefix="SiteCount_",
    title="Existing VRE Sites by Exclusion Classes and Technology",
    figsize=(8, 2),
    wrap_width=25,
    fontsize=7,
    # normalize=True,
    colors=["#2e0354", "#e0b732"],
    save_to=common_vis_save_to/"Existing_VREs_by_Exclusion_GAEZ.png"
)

- Define Layers to plot

In [ ]:
layers_excluded:dict=excld_cover_cfg['class_exclusion']

# Unique CLC codes in your clipped area
unique_classes = np.unique(excld_raster_data.values[~np.isnan(excld_raster_data.values)])
layers_included = {
    tech: [c for c in unique_classes if c not in layers_excluded[tech] and c != 0]
    for tech in ['solar', 'wind']
}

- Plot in Map

In [ ]:
classes_to_plot = layers_included #layers_included #None, for all classes

if classes_to_plot is None:
    title="Globally Protected Areas All Layers (GAEZ_v4 exclusion 2017)",
    save_to_plot = f"{common_vis_save_to}/GAEZ_exclusion_AllClasses_with_existing_VREs.png"
    utils.print_warning("[WARN] All classes are being plotted. This may lead to a cluttered visualization.")
else:
    title="Globally Protected Areas Selected Classes (GAEZ_v4 exclusion 2017)",
    save_to_plot = f"{POLICY_vis_save_to}/GAEZ_exclusion_SelectedClasses_with_existing_VREs.png"
    utils.print_update(level=1, message=f"Plotting selected classes: {layers_included}")
    
fig3,ax3,save_to3=vis.plot_developable_land_and_vres(
    target_crs=CRS_m,
    raster_data=excld_raster_data,
    raster_legends=excld_raster_legends,
    classes_to_plot=classes_to_plot, #layers_excluded, #None,                 # <- all classes
    boundary=boundary,
    existing_VREs_gdf=existing_VREs_gdf,
    committed_VREs_gdf=committed_VREs_gdf,
        marker_scale_existing=5.0,
    marker_scale_committed=0.4,
    title=title,
    dpi=500,
    figsize=(12, 12),
    existing_marker_col="wind_turbine_capacity",
    output_path=save_to_plot,
    legend_anchor=(0.73,0.89),
    marker_highlight_width=2,
)

utils.print_update(level=1,message=f"plot saved to:: {save_to_plot}")

## Vectors

### Aeroway

In [ ]:
aeroway=gpd.read_file(f'../data/downloaded_data/OSM/{region_code}_aeroway.geojson')
if aeroway.crs is None:
    aeroway.set_crs(CRS_m, allow_override=True, inplace=True)
if aeroway.crs != CRS_m:
    aeroway.to_crs(CRS_m, inplace=True)

* Load_policy_config

In [ ]:
cfg_for_plot=cfg_BASELINE

In [ ]:
aeroway_buffer_solar:list=cfg_for_plot.get('capacity_disaggregation').get('solar').get('vector_buffers')
aeroway_solar:dict= next(
    (item for item in aeroway_buffer_solar if item.get("aeroway")),
    None
)

aeroway_buffer_wind:list=cfg_for_plot.get('capacity_disaggregation').get('wind').get('vector_buffers')
aeroway_wind:dict= next(
    (item for item in aeroway_buffer_wind if item.get("aeroway")),
    None
)

In [ ]:
aeroway_buffer_solar_gdf,B1=lands.apply_buffer_to_vector(aeroway,CRS_m,CRS_d,aeroway_solar['aeroway']['buffer_mapping_key_buffers'],
                                 aeroway_solar['aeroway']['buffer_mapping_key'])


aeroway_buffer_wind_gdf,B2=lands.apply_buffer_to_vector(aeroway,CRS_m,CRS_d,aeroway_wind['aeroway']['buffer_mapping_key_buffers'],
                                 aeroway_wind['aeroway']['buffer_mapping_key'])

In [ ]:
if boundary.crs != CRS_m:
    boundary_proj=boundary.to_crs(CRS_m)
else:
    boundary_proj=boundary

if aeroway_buffer_solar_gdf.crs != CRS_m:
    aeroway_buffer_solar_gdf_proj=aeroway_buffer_solar_gdf.to_crs(CRS_m)
    aeroway_buffer_wind_gdf_proj=aeroway_buffer_wind_gdf.to_crs(CRS_m)
else:
    aeroway_buffer_solar_gdf_proj=aeroway_buffer_solar_gdf
    aeroway_buffer_wind_gdf_proj=aeroway_buffer_wind_gdf


In [ ]:

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import pandas as pd

# --- Collect all categories from both datasets ---
all_cats = pd.Index(
    pd.concat([
        aeroway_buffer_solar_gdf_proj["aeroway"],
        aeroway_buffer_wind_gdf_proj["aeroway"]
    ])
    .dropna()
    .unique()
)

# Build consistent color mapping
cmap =plt.get_cmap("prism", len(all_cats))
cat2color = {cat: cmap(i) for i, cat in enumerate(all_cats)}

# Map colors to each GeoDataFrame
aeroway_buffer_solar_gdf_proj["color"] = aeroway_buffer_solar_gdf_proj["aeroway"].map(cat2color)
aeroway_buffer_wind_gdf_proj["color"]  = aeroway_buffer_wind_gdf_proj["aeroway"].map(cat2color)

# --- Plot ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7.5, 4), dpi=500)
fig.suptitle(f"Aeroway Buffers (BASELINE)", weight='bold', fontsize=12,y=0.93)
# Solar buffers
boundary_proj.plot(ax=ax1, facecolor="grey", edgecolor="k", linewidth=0.4, alpha=0.1, zorder=3)
aeroway_buffer_solar_gdf_proj.centroid.plot(
    ax=ax1,
    color=aeroway_buffer_solar_gdf_proj["color"],
    edgecolor='None',
    markersize=aeroway_buffer_solar_gdf_proj["buffer_applied_m"]/300,
    alpha=0.4
)
ax1.set_title("No-go Buffers (Solar)",fontweight='bold',fontsize=10,y=0.95)
ax1.axis("off")

# Wind buffers
boundary_proj.plot(ax=ax2, facecolor="grey", edgecolor="k", linewidth=0.4, alpha=0.1, zorder=3)
aeroway_buffer_wind_gdf_proj.centroid.plot(
    ax=ax2,
    color=aeroway_buffer_wind_gdf_proj["color"],
    edgecolor='None',
    markersize=aeroway_buffer_wind_gdf_proj["buffer_applied_m"]/300,
    alpha=0.3
)
ax2.set_title("No-go Buffers (Wind)",fontweight='bold',fontsize=10,y=0.94)
ax2.axis("off")

# --- Shared Legend ---
handles = [
    mpatches.Patch(color=col, label=lab) for lab, col in cat2color.items()
]
fig.legend(handles=handles, title="Aeroway Category",
           loc="lower center", ncol=6, frameon=False, fontsize=10)

plt.tight_layout(rect=[0, 0.05, 1, 0.95])
plt.show()
fig.savefig(f'../vis/{country_kwd}/{region_code}/aeroway_buffers.svg', bbox_inches='tight')

save_to_root=f'{BASELINE_vis_save_to}' if cfg_for_plot['Scenario']['run_id']=='BASELINE' else f'{POLICY_vis_save_to}'
# Doc content
# fig.savefig('../docs/source/_static/aeroway_buffers.svg', bbox_inches='tight')
fig.savefig(f'{save_to_root}/aeroway_buffers.svg', bbox_inches='tight')

### CPCAD

In [ ]:
cpcad=pd.read_pickle(f'../data/downloaded_data/lands/ProtectedConservedArea_{region_code}.pickle')
if cpcad.crs is None:
    cpcad.set_crs(CRS_m, allow_override=True, inplace=True)
if cpcad.crs != CRS_m:
    cpcad.to_crs(CRS_m, inplace=True)

- Temp Fix (one time)

In [ ]:
# cpcad.to_pickle(f'../data/downloaded_data/lands/ProtectedConservedArea_{region_code}.pickle')
# # temp fix. New workflow should have this fixed.
# cpcad['IUCN_CAT_desc'] = cpcad['IUCN_CAT_desc'].replace(
#     {'Strict  Nature Reserve': 'Strict Nature Reserve'}
# )

In [ ]:
color_df = pd.read_csv("../data/legends/CPCAD_legends.csv")
cat2color = dict(zip(color_df["IUCN_CAT_desc"], color_df["color_hex"]))

In [ ]:
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import pandas as pd

# Shadow offset for boundary
shadow_offset = 0.008

fig, ax = plt.subplots(figsize=(10,7), dpi=1000)

# ---- 1) Build color column from cat2color safely ----
# Assume cat2color = {"Wilderness Area": "#1f77b4", "National Park": "#ff7f0e", ...}
cpcad["color"] = cpcad["IUCN_CAT_desc"].map(cat2color)

# Handle unmapped categories (assign gray fallback)
cpcad["color"] = cpcad["color"].fillna("#d3d3d3")

# Keep a stable order for legend
ordered_cats = pd.Index(cpcad["IUCN_CAT_desc"].dropna().unique()).tolist()

# ---- 2) CPCAD polygons with shadow ----
cpcad_shadow = cpcad.copy()
cpcad_shadow.geometry = cpcad_shadow.geometry.translate(
    xoff=-shadow_offset, yoff=shadow_offset
)
cpcad_shadow.plot(ax=ax, color=cpcad_shadow["color"], linewidth=0, alpha=0.35)

cpcad.plot(ax=ax, color=cpcad["color"], linewidth=0)

# ---- 3) VRE layers ----
ax, VRE_legend_handles =vis.get_existing_committed_VRE_plot(
    ax=ax,
    target_crs=CRS_m,
    existing_VREs_gdf=existing_VREs_gdf,
    committed_VREs_gdf=committed_VREs_gdf,
    marker_scale_existing=0.1,
    marker_scale_committed=0.25,
    sites_legend_handle_scale=6.0,
    marker_highlight_width=5.0
)

# ---- 4) Boundary with shadow ----
boundary_proj = boundary.to_crs(CRS_m) if boundary.crs != CRS_m else boundary
boundary_proj.plot(ax=ax, facecolor="none", edgecolor="gray", linewidth=0.3, alpha=1)
boundary_proj_shadow = boundary_proj.copy()
boundary_proj_shadow.geometry = boundary_proj_shadow.geometry.translate(
    xoff=shadow_offset, yoff=-shadow_offset
)
boundary_proj_shadow.plot(ax=ax, facecolor="none", edgecolor="gray", linewidth=0.2, alpha=0.9)

# ---- 5) Legend handles from cat2color (with fallback) ----
cpcad_handles = [
    mpatches.Patch(facecolor=cat2color.get(lab, "#d3d3d3"),
                   edgecolor="none", label=lab)
    for lab in ordered_cats
]

all_handles = cpcad_handles + VRE_legend_handles
ax.legend(
    handles=all_handles,
    # title="Protected Areas & VRE Projects",
    loc="upper left",
    bbox_to_anchor=(0.7, .9),
    frameon=False,
    prop={"size": 11}
)

# ---- 6) Final touches ----
ax.grid(False)
ax.axis("off")
plt.tight_layout()

# plt.savefig(f"../docs/source/_static/CPCAD_{region_code}.png", bbox_inches="tight")
plt.savefig(common_vis_save_to/f"CPCAD_{region_code}.png", bbox_inches="tight")

- Visualize scenario Buffers

In [ ]:
cfg_for_plot=cfg_BASELINE

In [ ]:
cpcad_buffer_solar:list=cfg_for_plot.get('capacity_disaggregation').get('solar').get('vector_buffers')
cpcad_solar:dict= next(
    (item for item in cpcad_buffer_solar if item.get("conserved_lands")),
    None
)

cpcad_buffer_wind:list=cfg_for_plot.get('capacity_disaggregation').get('wind').get('vector_buffers')
cpcad_wind:dict= next(
    (item for item in cpcad_buffer_wind if item.get("conserved_lands")),
    None
)

In [ ]:
cpcad_buffer_solar_gdf,B1=lands.apply_buffer_to_vector(cpcad,CRS_m,CRS_d,cpcad_solar['conserved_lands']['buffer_mapping_key_buffers'],cpcad_solar['conserved_lands']['buffer_mapping_key'])

cpcad_buffer_wind_gdf,B2=lands.apply_buffer_to_vector(cpcad,CRS_m,CRS_d,cpcad_wind['conserved_lands']['buffer_mapping_key_buffers'],cpcad_wind['conserved_lands']['buffer_mapping_key'])

In [ ]:
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import pandas as pd

# Shadow offset for boundary
shadow_offset = 0.008

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.5, 5), dpi=1000)
fig.suptitle(f"Canadian Protected and Conserved Areas (2023) [BASELINE]",
             fontsize=16, fontweight='bold', y=0.95)

# --- Ensure consistent CRS ---
cpcad_buf_solar = (cpcad_buffer_solar_gdf.to_crs(CRS_m)
                   if cpcad_buffer_solar_gdf.crs != CRS_m else cpcad_buffer_solar_gdf).copy()
cpcad_buf_wind  = (cpcad_buffer_wind_gdf.to_crs(CRS_m)
                   if cpcad_buffer_wind_gdf.crs != CRS_m else cpcad_buffer_wind_gdf).copy()
boundary_proj   = boundary.to_crs(CRS_m) if boundary.crs != CRS_m else boundary

# --- Apply cat2color mapping ---
# Load from CSV or define manually:
# color_df = pd.read_csv("CPCAD_color_codes.csv")
# cat2color = dict(zip(color_df["IUCN_CAT_desc"], color_df["color_hex"]))

cpcad_buf_solar["color"] = cpcad_buf_solar["IUCN_CAT_desc"].map(cat2color).fillna("#d3d3d3")
cpcad_buf_wind["color"]  = cpcad_buf_wind["IUCN_CAT_desc"].map(cat2color).fillna("#d3d3d3")

# --- Left: Solar buffer ---
cpcad_buf_solar.plot(color=cpcad_buf_solar["color"], ax=ax1, edgecolor="none")
boundary_proj.plot(ax=ax1, facecolor='none', edgecolor='gray', linewidth=0.3, alpha=1)
boundary_proj.translate(xoff=shadow_offset, yoff=-shadow_offset).plot(
    ax=ax1, facecolor='none', edgecolor='gray', linewidth=0.2, alpha=0.9
)
ax1.set_title("No-go Buffers (Solar)", fontsize=14, y=0.93)
ax1.axis('off')

# --- Right: Wind buffer ---
cpcad_buf_wind.plot(color=cpcad_buf_wind["color"], ax=ax2, edgecolor="none")
boundary_proj.plot(ax=ax2, facecolor='none', edgecolor='gray', linewidth=0.3, alpha=1)
boundary_proj.translate(xoff=shadow_offset, yoff=-shadow_offset).plot(
    ax=ax2, facecolor='none', edgecolor='gray', linewidth=0.2, alpha=0.9
)
ax2.set_title("No-go Buffers (Wind)", fontsize=14, y=0.93)
ax2.axis('off')

# --- Legend (from cat2color, ensures consistency) ---
handles = [mpatches.Patch(facecolor=color, edgecolor="none", label=cat)
           for cat, color in cat2color.items()]
fig.legend(handles=handles, title="IUCN Category",
           loc="lower center", ncol=3, frameon=False, fontsize=10)

# --- Layout tidy ---
plt.tight_layout(rect=[0, 0.05, 1, 0.95])
plt.savefig(f'../docs/source/_static/CPCAD_{region_code}_buffers.png', bbox_inches="tight")

save_to_root=f'{BASELINE_vis_save_to}' if cfg_for_plot['Scenario']['run_id']=='BASELINE' else f'{POLICY_vis_save_to}'
plt.savefig(f'{save_to_root}/CPCAD_buffers_side_by_side_{region_code}.svg', bbox_inches="tight")